In [1]:
import os
print(os.getcwd())


E:\Governance-Policy


In [2]:
import os
import zipfile

ZIP_PATH = r"E:\Governance-Policy\wb3ndt69gf-3.zip"
EXTRACT_DIR = r"E:\Governance-Policy\wb3ndt69gf-3_extracted"

print("ZIP exists:", os.path.exists(ZIP_PATH))

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("\nFiles inside ZIP:")
    for name in z.namelist():
        print(name)

ZIP exists: True

Files inside ZIP:
Labels.do
P2P_Macro_Codes.do
P2P_Macro_Data.zip
P2P_Macro_Data.dta


In [3]:
import os
import zipfile

ZIP_PATH = r"E:\Governance-Policy\wb3ndt69gf-3.zip"
EXTRACT_DIR = r"E:\Governance-Policy\wb3ndt69gf-3_extracted"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extract("P2P_Macro_Data.dta", EXTRACT_DIR)

DTA_PATH = os.path.join(EXTRACT_DIR, "P2P_Macro_Data.dta")

print("DTA exists:", os.path.exists(DTA_PATH))
print("DTA path:", DTA_PATH)
print("File size (GB):", round(os.path.getsize(DTA_PATH) / (1024**3), 3))

DTA exists: True
DTA path: E:\Governance-Policy\wb3ndt69gf-3_extracted\P2P_Macro_Data.dta
File size (GB): 2.814


In [4]:
import pandas as pd

reader = pd.read_stata(
    DTA_PATH,
    convert_categoricals=False,
    chunksize=5
)

sample = next(reader)

print("Number of columns:", len(sample.columns))

keywords = [
    "issue", "status", "term", "last", "pymnt",
    "default", "badloan", "charge", "date", "unem"
]

relevant_cols = [
    c for c in sample.columns
    if any(k in c.lower() for k in keywords)
]

print("\nPotentially relevant columns:")
for c in relevant_cols:
    print(c)

Number of columns: 195

Potentially relevant columns:
term
verification_status
issue_d
loan_status
pymnt_plan
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
initial_list_status
total_pymnt
total_pymnt_inv
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
mths_since_last_major_derog
verification_status_joint
inq_last_12m
chargeoff_within_12_mths
issue_date
last_date
unemployment
unem_rate
initial_list_status_
pymnt_plan_
verification_status_
term_
loan_status_
badloan
rankloan_status_
logunemployment
daily_date
loan_status_5
issue_daily
issue_year


In [5]:
# ============================================================
# W1 - Step 2: Inspect key outcome and timing variables
# ============================================================

key_cols = [
    "term",
    "issue_d",
    "issue_date",
    "loan_status",
    "loan_status_",
    "loan_status_5",
    "badloan",
    "last_pymnt_d",
    "last_date",
    "chargeoff_within_12_mths",
    "unemployment",
    "unem_rate"
]

reader = pd.read_stata(
    DTA_PATH,
    columns=key_cols,
    convert_categoricals=False,
    chunksize=1000
)

check = next(reader)

print("Shape of inspection sample:", check.shape)

for c in key_cols:
    print("\n" + "=" * 70)
    print("COLUMN:", c)
    print("dtype:", check[c].dtype)
    print("Missing:", check[c].isna().sum())
    print("First 15 unique values:")
    print(check[c].dropna().unique()[:15])

Shape of inspection sample: (1000, 12)

COLUMN: term
dtype: object
Missing: 0
First 15 unique values:
[' 36 months' ' 60 months']

COLUMN: issue_d
dtype: object
Missing: 0
First 15 unique values:
['Aug-12' 'Sep-12' 'Oct-12' 'Nov-12' 'Dec-12' 'Jan-13' 'Feb-13' 'Mar-13'
 'Apr-13' 'May-13' 'Jun-13' 'Jul-13' 'Aug-13' 'Sep-13' 'Oct-13']

COLUMN: issue_date
dtype: datetime64[ns]
Missing: 0
First 15 unique values:
<DatetimeArray>
['2012-08-01 00:00:00', '2012-09-01 00:00:00', '2012-10-01 00:00:00',
 '2012-11-01 00:00:00', '2012-12-01 00:00:00', '2013-01-01 00:00:00',
 '2013-02-01 00:00:00', '2013-03-01 00:00:00', '2013-04-01 00:00:00',
 '2013-05-01 00:00:00', '2013-06-01 00:00:00', '2013-07-01 00:00:00',
 '2013-08-01 00:00:00', '2013-09-01 00:00:00', '2013-10-01 00:00:00']
Length: 15, dtype: datetime64[ns]

COLUMN: loan_status
dtype: object
Missing: 0
First 15 unique values:
['Fully Paid' 'Charged Off' 'Current' 'Default' 'Late (31-120 days)'
 'In Grace Period' 'Late (16-30 days)']

COLUMN: l

In [6]:
# ============================================================
# W1 - Step 3: Inspect original Stata variable labels
# ============================================================

from pandas.io.stata import StataReader

meta_reader = StataReader(
    DTA_PATH,
    convert_categoricals=False
)

var_labels = meta_reader.variable_labels()

fields = [
    "term",
    "issue_d",
    "issue_date",
    "loan_status",
    "loan_status_",
    "loan_status_5",
    "badloan",
    "last_pymnt_d",
    "last_date",
    "chargeoff_within_12_mths",
    "unemployment",
    "unem_rate"
]

for c in fields:
    print(f"{c:30s} : {var_labels.get(c, 'NO LABEL FOUND')}")

term                           : The number of payments on the loan. Values are in months and can be either 36 or
issue_d                        : The month which the loan was funded
issue_date                     : 
loan_status                    : Current status of the loan
loan_status_                   : 
loan_status_5                  : RECODE of rankloan_status_ (ranked loan status)
badloan                        : 
last_pymnt_d                   : Last month payment was received
last_date                      : 
chargeoff_within_12_mths       : Number of charge-offs within 12 months
unemployment                   : 
unem_rate                      : 


In [7]:
# ============================================================
# W1 - Step 4: Verify outcome mapping and timing fields
# ============================================================

cols = [
    "issue_date",
    "loan_status",
    "badloan",
    "last_pymnt_d",
    "last_date",
    "term"
]

reader = pd.read_stata(
    DTA_PATH,
    columns=cols,
    convert_categoricals=False,
    chunksize=100_000
)

tmp = next(reader)

print("Sample size:", len(tmp))

print("\nBADLOAN MAPPING BY LOAN STATUS")
print(
    pd.crosstab(
        tmp["loan_status"],
        tmp["badloan"],
        margins=True
    )
)

# Parse original last payment text for comparison
parsed_last_pymnt = pd.to_datetime(
    tmp["last_pymnt_d"],
    format="%b-%y",
    errors="coerce"
)

same_date = (
    parsed_last_pymnt.eq(tmp["last_date"]) |
    (parsed_last_pymnt.isna() & tmp["last_date"].isna())
)

print("\nLAST_DATE CHECK")
print("Rows where parsed last_pymnt_d == last_date:",
      same_date.sum(), "/", len(tmp))
print("Agreement rate:", round(same_date.mean(), 6))

# Observed months from origination to last payment
months_observed = (
    (tmp["last_date"].dt.year - tmp["issue_date"].dt.year) * 12
    + (tmp["last_date"].dt.month - tmp["issue_date"].dt.month)
)

print("\nMONTHS FROM ISSUE_DATE TO LAST_DATE BY STATUS")
print(
    tmp.assign(months_observed=months_observed)
       .groupby("loan_status")["months_observed"]
       .agg(["count", "min", "median", "max"])
       .sort_index()
)

Sample size: 100000

BADLOAN MAPPING BY LOAN STATUS
badloan               0.0    1.0     All
loan_status                             
Charged Off             0   8653    8653
Current             55838      0   55838
Default                 0      4       4
Fully Paid          33311      0   33311
In Grace Period       641      0     641
Late (16-30 days)       0    275     275
Late (31-120 days)      0   1278    1278
All                 89790  10210  100000

LAST_DATE CHECK
Rows where parsed last_pymnt_d == last_date: 99901 / 100000
Agreement rate: 0.99901

MONTHS FROM ISSUE_DATE TO LAST_DATE BY STATUS
                    count  min  median  max
loan_status                                
Charged Off          8653    0    14.0   49
Current             55838    1     9.0   56
Default                 4    7     8.0   36
Fully Paid          33311    0    19.0   53
In Grace Period       641    1    10.0   52
Late (16-30 days)     275    0    10.0   51
Late (31-120 days)   1278    0    10.0

In [8]:
# ============================================================
# W1 - Step 5: Audit 36-month loans by origination year/status
# ============================================================

import pandas as pd

audit_parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status",
        "last_date"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    # Standardise term
    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    # Keep 36-month loans only
    x = chunk.loc[chunk["term_months"] == 36].copy()

    x["issue_year"] = x["issue_date"].dt.year

    # Focus on vintages potentially fully observable by Dec 2019
    x = x.loc[x["issue_year"].between(2012, 2016)]

    audit_parts.append(
        x.groupby(["issue_year", "loan_status"])
         .size()
         .rename("n")
         .reset_index()
    )

    print(f"Finished chunk {i}")

audit = (
    pd.concat(audit_parts, ignore_index=True)
      .groupby(["issue_year", "loan_status"], as_index=False)["n"]
      .sum()
)

print("\n36-MONTH LOANS: STATUS BY ISSUE YEAR")
print(
    audit.pivot(
        index="issue_year",
        columns="loan_status",
        values="n"
    ).fillna(0).astype(int)
)

print("\nTOTAL 36-MONTH LOANS BY ISSUE YEAR")
print(
    audit.groupby("issue_year")["n"].sum()
)

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

36-MONTH LOANS: STATUS BY ISSUE YEAR
loan_status  Charged Off  Current  Default  Fully Paid  In Grace Period  \
issue_year                                                                
2012                2717        0        0       18193                0   
2013               12359        0        2       87904                1   
2014               19841    43228       15       97037              785   
2015               42160       12        0      240924                1   
2016                7531   268313       48       37890             3003   

loan_status  Late (16-30 days)  Late (31-120 days)  
issue_year                                          
2012                         0                   0  
2013                         0                   3  
2014                       278     

In [9]:
# ============================================================
# W1 - Step 6: Check observation timing by year and status
# ============================================================

timing_parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status",
        "last_date"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    x = chunk.loc[chunk["term_months"] == 36].copy()

    x["issue_year"] = x["issue_date"].dt.year

    x = x.loc[x["issue_year"].between(2012, 2016)]

    timing_parts.append(
        x.groupby(["issue_year", "loan_status"])["last_date"]
         .agg(["count", "min", "median", "max"])
         .reset_index()
    )

    print(f"Finished chunk {i}")

timing = pd.concat(timing_parts, ignore_index=True)

# Re-aggregate correctly across chunks
summary = (
    timing.groupby(["issue_year", "loan_status"])
    .agg(
        n=("count", "sum"),
        earliest_last_date=("min", "min"),
        latest_last_date=("max", "max")
    )
    .reset_index()
)

print("\nOBSERVATION TIMING BY ISSUE YEAR AND STATUS")
print(summary.to_string(index=False))

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

OBSERVATION TIMING BY ISSUE YEAR AND STATUS
 issue_year        loan_status      n earliest_last_date latest_last_date
       2012        Charged Off   2717         2012-08-01       2016-04-01
       2012         Fully Paid  18193         2012-09-01       2017-02-01
       2013        Charged Off  12359         2013-01-01       2017-02-01
       2013            Default      2         2016-11-01       2016-12-01
       2013         Fully Paid  87904         2013-02-01       2017-04-01
       2013    In Grace Period      1         2017-04-01       2017-04-01
       2013 Late (31-120 days)      3         2017-03-01       2017-04-01
       2014        Charged Off  19841         2014-01-01       2017-04-01
       2014            Current  43228         2017-03-01       2017-04-01
       2014            Def

In [11]:
# ============================================================
# W1 - Step 7: Inspect dataset construction code/documentation
# ============================================================

import zipfile
import re

ZIP_PATH = r"E:\Governance-Policy\wb3ndt69gf-3.zip"

search_terms = [
    "last_date",
    "last_pymnt",
    "badloan",
    "loan_status",
    "issue_date",
    "charge",
    "default"
]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for filename in ["P2P_Macro_Codes.do", "Labels.do"]:
        print("\n" + "=" * 80)
        print("FILE:", filename)
        print("=" * 80)

        text = z.read(filename).decode("utf-8", errors="ignore")
        lines = text.splitlines()

        for i, line in enumerate(lines, start=1):
            if any(term.lower() in line.lower() for term in search_terms):
                print(f"{i:5d}: {line}")


FILE: P2P_Macro_Codes.do
   11: gen issue_date=monthly(issue_d, "MY",2050)
   12: format %tmMonth_CCYY issue_date
   13: gen last_date=monthly(last_pymnt_d, "MY",2050)
   14: format %tmMonth_CCYY last_date
   15: replace last_date=issue_date if last_date==.
   16: gen daily_date=dofm(last_date)
   21: gen issue_daily=dofm(issue_date)
   32: gen loan_status_=loan_status
   33: replace loan_status_="Fully Paid" if loan_status=="Does not meet the credit policy. Status:Fully Paid"
   34: replace loan_status_="Charged Off" if loan_status=="Does not meet the credit policy. Status:Charged Off"
   35: gen badloan=1 if loan_status=="Charged Off" | loan_status=="Default" | loan_status=="Does not meet the credit policy. Status:Charged Off" | ///
   36: 	loan_status=="Late (16-30 days)" | loan_status=="Late (31-120 days)"
   37: replace badloan=0 if badloan==.
   38: encode loan_status_, gen(rankloan_status_)
   54: summarize badloan inf muni_points logfunded_amnt  int_rate inq_last_6mths open_ac

In [12]:
# ============================================================
# W1 - Step 8: Identify apparent dataset snapshot dates
# using CURRENT loans
# ============================================================

snapshot_parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status",
        "last_date"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    x = chunk[
        (chunk["term_months"] == 36) &
        (chunk["loan_status"] == "Current")
    ].copy()

    x["issue_year"] = x["issue_date"].dt.year

    x = x[x["issue_year"].between(2014, 2016)]

    snapshot_parts.append(
        x.groupby(["issue_year", "last_date"])
         .size()
         .rename("n")
         .reset_index()
    )

    print(f"Finished chunk {i}")

snapshot_counts = (
    pd.concat(snapshot_parts, ignore_index=True)
      .groupby(["issue_year", "last_date"], as_index=False)["n"]
      .sum()
)

for year in [2014, 2015, 2016]:
    print("\n" + "=" * 70)
    print(f"TOP LAST_DATE VALUES FOR CURRENT 36-MONTH LOANS: {year}")
    print("=" * 70)

    out = (
        snapshot_counts[snapshot_counts["issue_year"] == year]
        .sort_values("n", ascending=False)
        .head(15)
    )

    print(out.to_string(index=False))

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

TOP LAST_DATE VALUES FOR CURRENT 36-MONTH LOANS: 2014
 issue_year  last_date     n
       2014 2017-04-01 40413
       2014 2017-03-01  2815

TOP LAST_DATE VALUES FOR CURRENT 36-MONTH LOANS: 2015
 issue_year  last_date  n
       2015 2019-05-01  9
       2015 2018-07-01  2
       2015 2018-10-01  1

TOP LAST_DATE VALUES FOR CURRENT 36-MONTH LOANS: 2016
 issue_year  last_date      n
       2016 2017-04-01 242898
       2016 2017-03-01  25415


In [13]:
# ============================================================
# W1 - Step 9: 2014 36-month loans by issue month and status
# ============================================================

parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    x = chunk[
        (chunk["term_months"] == 36) &
        (chunk["issue_date"].dt.year == 2014)
    ].copy()

    x["issue_month"] = x["issue_date"].dt.to_period("M").astype(str)

    parts.append(
        x.groupby(["issue_month", "loan_status"])
         .size()
         .rename("n")
         .reset_index()
    )

    print(f"Finished chunk {i}")

monthly = (
    pd.concat(parts, ignore_index=True)
      .groupby(["issue_month", "loan_status"], as_index=False)["n"]
      .sum()
)

table = (
    monthly.pivot(
        index="issue_month",
        columns="loan_status",
        values="n"
    )
    .fillna(0)
    .astype(int)
)

table["TOTAL"] = table.sum(axis=1)

print("\n2014 36-MONTH LOANS BY ISSUE MONTH AND FINAL STATUS")
print(table.to_string())

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

2014 36-MONTH LOANS BY ISSUE MONTH AND FINAL STATUS
loan_status  Charged Off  Current  Default  Fully Paid  In Grace Period  Late (16-30 days)  Late (31-120 days)  TOTAL
issue_month                                                                                                          
2014-01             1420        0        3       10046                0                  0                  19  11488
2014-02             1387        1        0        9514                0                  0                  37  10939
2014-03             1484       61        0       10016                1                  4                  63  11629
2014-04             1697      766        1       10633               61                 17                  85  13260
2014-05             1641     3888        0        

In [14]:
# ============================================================
# W1 - Step 10: 2015 36-month loans by issue month and status
# ============================================================

parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    x = chunk[
        (chunk["term_months"] == 36) &
        (chunk["issue_date"].dt.year == 2015)
    ].copy()

    x["issue_month"] = x["issue_date"].dt.to_period("M").astype(str)

    parts.append(
        x.groupby(["issue_month", "loan_status"])
         .size()
         .rename("n")
         .reset_index()
    )

    print(f"Finished chunk {i}")

monthly_2015 = (
    pd.concat(parts, ignore_index=True)
      .groupby(["issue_month", "loan_status"], as_index=False)["n"]
      .sum()
)

table_2015 = (
    monthly_2015.pivot(
        index="issue_month",
        columns="loan_status",
        values="n"
    )
    .fillna(0)
    .astype(int)
)

table_2015["TOTAL"] = table_2015.sum(axis=1)

# unresolved = still active / delinquent but not terminal
unresolved_cols = [
    c for c in [
        "Current",
        "In Grace Period",
        "Late (16-30 days)",
        "Late (31-120 days)"
    ]
    if c in table_2015.columns
]

table_2015["UNRESOLVED"] = table_2015[unresolved_cols].sum(axis=1)
table_2015["UNRESOLVED_SHARE"] = (
    table_2015["UNRESOLVED"] / table_2015["TOTAL"]
)

print("\n2015 36-MONTH LOANS BY ISSUE MONTH AND FINAL STATUS")
print(table_2015.to_string())

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

2015 36-MONTH LOANS BY ISSUE MONTH AND FINAL STATUS
loan_status  Charged Off  Current  Fully Paid  In Grace Period  Late (31-120 days)  TOTAL  UNRESOLVED  UNRESOLVED_SHARE
issue_month                                                                                                            
2015-01             3515        0       20110                0                   0  23625           0          0.000000
2015-02             2270        0       13751                0                   0  16021           0          0.000000
2015-03             2605        0       14308                0                   0  16913           0          0.000000
2015-04             3668        0       19944                0                   0  23612           0          0.000000
2015-05             3278        0     

In [15]:
# ============================================================
# W1 - Step 11: Check follow-up of unresolved 2015 36-month loans
# ============================================================

unresolved_parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status",
        "last_date"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

unresolved_statuses = [
    "Current",
    "In Grace Period",
    "Late (16-30 days)",
    "Late (31-120 days)"
]

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    x = chunk[
        (chunk["term_months"] == 36) &
        (chunk["issue_date"].dt.year == 2015) &
        (chunk["loan_status"].isin(unresolved_statuses))
    ].copy()

    x["followup_months"] = (
        (x["last_date"].dt.year - x["issue_date"].dt.year) * 12
        + (x["last_date"].dt.month - x["issue_date"].dt.month)
    )

    unresolved_parts.append(x)

    print(f"Finished chunk {i}")

unresolved_2015 = pd.concat(unresolved_parts, ignore_index=True)

print("\nNUMBER OF UNRESOLVED 2015 LOANS:")
print(len(unresolved_2015))

print("\nUNRESOLVED LOANS:")
print(
    unresolved_2015[
        ["issue_date", "loan_status", "last_date", "followup_months"]
    ]
    .sort_values(["issue_date", "loan_status"])
    .to_string(index=False)
)

print("\nFOLLOW-UP SUMMARY:")
print(unresolved_2015["followup_months"].describe())

print(
    "\nNumber with >= 36 months follow-up:",
    (unresolved_2015["followup_months"] >= 36).sum()
)

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

NUMBER OF UNRESOLVED 2015 LOANS:
24

UNRESOLVED LOANS:
issue_date        loan_status  last_date  followup_months
2015-07-01            Current 2018-07-01               36
2015-09-01            Current 2018-07-01               34
2015-10-01            Current 2018-10-01               36
2015-10-01 Late (31-120 days) 2019-05-01               43
2015-10-01 Late (31-120 days) 2019-03-01               41
2015-11-01 Late (31-120 days) 2019-05-01               42
2015-12-01            Current 2019-05-01               41
2015-12-01            Current 2019-05-01               41
2015-12-01            Current 2019-05-01               41
2015-12-01            Current 2019-05-01               41
2015-12-01            Current 2019-05-01               41
2015-12-01            Current 2019-05-01               41
2

In [16]:
# ============================================================
# W1 - Step 12: Check timing of Charged Off 2015 36-month loans
# ============================================================

parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status",
        "last_date"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    x = chunk[
        (chunk["term_months"] == 36) &
        (chunk["issue_date"].dt.year == 2015) &
        (chunk["loan_status"] == "Charged Off")
    ].copy()

    x["followup_months"] = (
        (x["last_date"].dt.year - x["issue_date"].dt.year) * 12
        + (x["last_date"].dt.month - x["issue_date"].dt.month)
    )

    parts.append(x[["issue_date", "last_date", "followup_months"]])

    print(f"Finished chunk {i}")

charged_2015 = pd.concat(parts, ignore_index=True)

print("\nNUMBER OF CHARGED OFF LOANS:")
print(len(charged_2015))

print("\nFOLLOW-UP SUMMARY:")
print(charged_2015["followup_months"].describe())

print("\nCOUNTS BY FOLLOW-UP RANGE:")
print("<= 12 months:", (charged_2015["followup_months"] <= 12).sum())
print("<= 24 months:", (charged_2015["followup_months"] <= 24).sum())
print("<= 36 months:", (charged_2015["followup_months"] <= 36).sum())
print("> 36 months:",  (charged_2015["followup_months"] > 36).sum())

print("\nMAXIMUM FOLLOW-UP EXAMPLES:")
print(
    charged_2015
    .sort_values("followup_months", ascending=False)
    .head(20)
    .to_string(index=False)
)

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

NUMBER OF CHARGED OFF LOANS:
42160

FOLLOW-UP SUMMARY:
count    42160.000000
mean        16.952253
std          8.848205
min          0.000000
25%         10.000000
50%         16.000000
75%         23.000000
max         43.000000
Name: followup_months, dtype: float64

COUNTS BY FOLLOW-UP RANGE:
<= 12 months: 14908
<= 24 months: 32842
<= 36 months: 41851
> 36 months: 309

MAXIMUM FOLLOW-UP EXAMPLES:
issue_date  last_date  followup_months
2015-04-01 2018-11-01               43
2015-04-01 2018-11-01               43
2015-07-01 2019-01-01               42
2015-09-01 2019-02-01               41
2015-06-01 2018-11-01               41
2015-10-01 2019-03-01               41
2015-07-01 2018-12-01               41
2015-01-01 2018-06-01               41
2015-08-01 2019-01-01               41
2015-05-01 2018-0

In [17]:
# ============================================================
# W1 - Step 13: Search metadata for possible event-date fields
# ============================================================

from pandas.io.stata import StataReader

meta_reader = StataReader(
    DTA_PATH,
    convert_categoricals=False
)

var_labels = meta_reader.variable_labels()

keywords = [
    "charge",
    "default",
    "settlement",
    "hardship",
    "payment",
    "pymnt",
    "status",
    "date"
]

print("POSSIBLE OUTCOME / EVENT-TIMING VARIABLES\n")

for c in sample.columns:
    label = var_labels.get(c, "")
    
    combined = f"{c} {label}".lower()
    
    if any(k in combined for k in keywords):
        print(f"{c:35s} : {label}")

POSSIBLE OUTCOME / EVENT-TIMING VARIABLES

term                                : The number of payments on the loan. Values are in months and can be either 36 or
installment                         : The monthly payment owed by the borrower if the loan originates.
home_ownership                      : The home ownership status provided by the borrower during registration or obtain
verification_status                 : Indicates if income was verified by LC, not verified, or if the income source wa
loan_status                         : Current status of the loan
pymnt_plan                          : Indicates if a payment plan has been put in place for the loan
dti                                 : A ratio calculated using the borrower’s total monthly debt payments on the total
initial_list_status                 : The initial listing status of the loan. Possible values are – W, F
total_pymnt                         : Payments received to date for total amount funded
total_pymnt_inv    

In [18]:
# ============================================================
# W1 - Step 14: Audit candidate fully observed 36-month cohorts
# ============================================================

parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status",
        "last_date"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    x = chunk[chunk["term_months"] == 36].copy()

    x["followup_months"] = (
        (x["last_date"].dt.year - x["issue_date"].dt.year) * 12
        + (x["last_date"].dt.month - x["issue_date"].dt.month)
    )

    # Proposed chronological cohorts
    x["split_candidate"] = pd.NA

    # Train: Aug 2012 - Dec 2013
    x.loc[
        (x["issue_date"] >= "2012-08-01") &
        (x["issue_date"] <  "2014-01-01"),
        "split_candidate"
    ] = "Train"

    # Validation: Jan-Mar 2014
    x.loc[
        (x["issue_date"] >= "2014-01-01") &
        (x["issue_date"] <  "2014-04-01"),
        "split_candidate"
    ] = "Validation"

    # Test: full calendar year 2015
    x.loc[
        (x["issue_date"] >= "2015-01-01") &
        (x["issue_date"] <  "2016-01-01"),
        "split_candidate"
    ] = "Test"

    x = x[x["split_candidate"].notna()].copy()

    x["default_final"] = x["loan_status"].isin(
        ["Charged Off", "Default"]
    ).astype(int)

    x["terminal"] = x["loan_status"].isin(
        ["Fully Paid", "Charged Off", "Default"]
    )

    # A non-terminal loan needs >=36 months follow-up
    # to have a usable contractual-horizon outcome.
    x["usable_36m"] = (
        x["terminal"] |
        (x["followup_months"] >= 36)
    )

    parts.append(
        x[
            [
                "split_candidate",
                "loan_status",
                "default_final",
                "terminal",
                "followup_months",
                "usable_36m"
            ]
        ]
    )

    print(f"Finished chunk {i}")

audit36 = pd.concat(parts, ignore_index=True)

print("\nCANDIDATE COHORT SUMMARY")
summary = (
    audit36.groupby("split_candidate")
    .agg(
        total=("default_final", "size"),
        usable_36m=("usable_36m", "sum"),
        defaults=("default_final", "sum"),
        default_rate=("default_final", "mean")
    )
)

summary["excluded_incomplete"] = (
    summary["total"] - summary["usable_36m"]
)

print(summary)

print("\nINCOMPLETE (<36 MONTH) NON-TERMINAL LOANS")
print(
    audit36.loc[
        ~audit36["usable_36m"],
        ["split_candidate", "loan_status", "followup_months"]
    ]
    .value_counts()
    .sort_index()
)

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

CANDIDATE COHORT SUMMARY
                  total  usable_36m  defaults  default_rate  \
split_candidate                                               
Test             283108      283107     42160      0.148918   
Train            121179      121179     15078      0.124427   
Validation        34056       33988      4294      0.126086   

                 excluded_incomplete  
split_candidate                       
Test                               1  
Train                              0  
Validation                        68  

INCOMPLETE (<36 MONTH) NON-TERMINAL LOANS
split_candidate  loan_status         followup_months
Test             Current             34                  1
Validation       Late (31-120 days)  33                 16
                                     34                 18
 

In [19]:
# ============================================================
# W1 - Step 15: Audit clean fixed-vintage 36-month design
# ============================================================

parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status",
        "last_date"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    # 36-month contracts only
    x = chunk[chunk["term_months"] == 36].copy()

    x["split"] = pd.NA

    # Training: Aug 2012 - Dec 2013
    x.loc[
        (x["issue_date"] >= "2012-08-01") &
        (x["issue_date"] <  "2014-01-01"),
        "split"
    ] = "Train"

    # Validation: Jan - Apr 2015
    x.loc[
        (x["issue_date"] >= "2015-01-01") &
        (x["issue_date"] <  "2015-05-01"),
        "split"
    ] = "Validation"

    # Test: May - Aug 2015
    x.loc[
        (x["issue_date"] >= "2015-05-01") &
        (x["issue_date"] <  "2015-09-01"),
        "split"
    ] = "Test"

    x = x[x["split"].notna()].copy()

    x["followup_months"] = (
        (x["last_date"].dt.year - x["issue_date"].dt.year) * 12
        + (x["last_date"].dt.month - x["issue_date"].dt.month)
    )

    # Terminal outcome observed directly
    x["terminal"] = x["loan_status"].isin(
        ["Fully Paid", "Charged Off", "Default"]
    )

    # Usable without outcome-dependent selection:
    # terminal outcome OR at least 36 months observed
    x["usable_36m"] = (
        x["terminal"] |
        (x["followup_months"] >= 36)
    )

    # Conference-paper default definition
    x["default"] = x["loan_status"].isin(
        ["Charged Off", "Default"]
    ).astype(int)

    parts.append(x)

    print(f"Finished chunk {i}")

fixed36_audit = pd.concat(parts, ignore_index=True)

print("\nFIXED-VINTAGE 36-MONTH AUDIT")
summary = (
    fixed36_audit
    .groupby("split")
    .agg(
        n=("default", "size"),
        usable=("usable_36m", "sum"),
        defaults=("default", "sum"),
        default_rate=("default", "mean")
    )
)

summary["incomplete"] = summary["n"] - summary["usable"]

print(summary)

print("\nSTATUS COUNTS BY SPLIT")
print(
    pd.crosstab(
        fixed36_audit["split"],
        fixed36_audit["loan_status"]
    )
)

print("\nANY INCOMPLETE RECORDS?")
print(
    fixed36_audit.loc[
        ~fixed36_audit["usable_36m"],
        ["split", "issue_date", "loan_status",
         "last_date", "followup_months"]
    ].to_string(index=False)
)

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

FIXED-VINTAGE 36-MONTH AUDIT
                 n  usable  defaults  default_rate  incomplete
split                                                         
Test         95389   95389     14127      0.148099           0
Train       121179  121179     15078      0.124427           0
Validation   80171   80171     12058      0.150404           0

STATUS COUNTS BY SPLIT
loan_status  Charged Off  Current  Default  Fully Paid  In Grace Period  \
split                                                                     
Test               14127        1        0       81261                0   
Train              15076        0        2      106097                1   
Validation         12058        0        0       68113                0   

loan_status  Late (31-120 days)  
split                           

In [20]:
# ============================================================
# W1 - Step 16: Inspect candidate predictor columns
# and identify obvious post-origination leakage fields
# ============================================================

all_cols = list(sample.columns)

leakage_keywords = [
    "pymnt",
    "recover",
    "collection_recovery",
    "last_",
    "next_",
    "loan_status",
    "badloan",
    "out_prncp",
    "total_rec",
    "settlement",
    "hardship"
]

obvious_leakage = [
    c for c in all_cols
    if any(k in c.lower() for k in leakage_keywords)
]

print("TOTAL RAW COLUMNS:", len(all_cols))

print("\nOBVIOUS POST-ORIGINATION / OUTCOME-RELATED COLUMNS:")
for c in obvious_leakage:
    print(c)

print("\nNUMBER FLAGGED:", len(obvious_leakage))

TOTAL RAW COLUMNS: 195

OBVIOUS POST-ORIGINATION / OUTCOME-RELATED COLUMNS:
loan_status
pymnt_plan
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
mths_since_last_major_derog
inq_last_12m
last_date
pymnt_plan_
loan_status_
badloan
rankloan_status_
loan_status_5

NUMBER FLAGGED: 26


In [21]:
# ============================================================
# W1 - Step 17: Audit the curated origination-time feature set
# from the parent-study notebook
# ============================================================

borrower_features = [
    "annual_inc",
    "dti",
    "emp_length",
    "home_ownership",
    "verification_status",
    "delinq_2yrs",
    "pub_rec",
    "mort_acc",
    "tax_liens",
    "pub_rec_bankruptcies"
]

loan_features = [
    "loan_amnt",
    "funded_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "purpose",
    "initial_list_status"
]

credit_features = [
    "open_acc",
    "total_acc",
    "revol_bal",
    "revol_util",
    "pct_tl_nvr_dlq",
    "num_rev_accts",
    "num_sats",
    "tot_hi_cred_lim",
    "total_bc_limit"
]

macro_features = [
    "unemployment",
    "unem_rate",
    "gdpindex",
    "gdpreal",
    "gdpcontrib",
    "cpi",
    "inf",
    "FEDFUNDS",
    "gdppercap",
    "riskprem"
]

time_features = [
    "issue_year",
    "quarter",
    "month"
]

planned_features = (
    borrower_features
    + loan_features
    + credit_features
    + macro_features
    + time_features
)

existing_features = [
    c for c in planned_features
    if c in sample.columns
]

missing_features = [
    c for c in planned_features
    if c not in sample.columns
]

print("Planned raw features:", len(planned_features))
print("Existing in dataset:", len(existing_features))
print("Missing:", len(missing_features))

print("\nMISSING FEATURES:")
for c in missing_features:
    print(c)

print("\nEXISTING FEATURES:")
for c in existing_features:
    print(c)

Planned raw features: 41
Existing in dataset: 41
Missing: 0

MISSING FEATURES:

EXISTING FEATURES:
annual_inc
dti
emp_length
home_ownership
verification_status
delinq_2yrs
pub_rec
mort_acc
tax_liens
pub_rec_bankruptcies
loan_amnt
funded_amnt
term
int_rate
installment
grade
sub_grade
purpose
initial_list_status
open_acc
total_acc
revol_bal
revol_util
pct_tl_nvr_dlq
num_rev_accts
num_sats
tot_hi_cred_lim
total_bc_limit
unemployment
unem_rate
gdpindex
gdpreal
gdpcontrib
cpi
inf
FEDFUNDS
gdppercap
riskprem
issue_year
quarter
month


In [22]:
# ============================================================
# W1 - Step 18: Verify clean 2015-only fixed-horizon split
# ============================================================

parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "loan_status",
        "last_date"
    ],
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    chunk["term_months"] = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    x = chunk[
        (chunk["term_months"] == 36) &
        (chunk["issue_date"].dt.year == 2015)
    ].copy()

    x["split"] = pd.NA

    # Train: Jan-Jun 2015
    x.loc[
        (x["issue_date"] >= "2015-01-01") &
        (x["issue_date"] <  "2015-07-01"),
        "split"
    ] = "Train"

    # Validation: Jul-Aug 2015
    x.loc[
        (x["issue_date"] >= "2015-07-01") &
        (x["issue_date"] <  "2015-09-01"),
        "split"
    ] = "Validation"

    # September omitted as an entire vintage month

    # Test: Oct-Dec 2015
    x.loc[
        (x["issue_date"] >= "2015-10-01") &
        (x["issue_date"] <  "2016-01-01"),
        "split"
    ] = "Test"

    x = x[x["split"].notna()].copy()

    x["followup_months"] = (
        (x["last_date"].dt.year - x["issue_date"].dt.year) * 12
        + (x["last_date"].dt.month - x["issue_date"].dt.month)
    )

    x["terminal"] = x["loan_status"].isin(
        ["Fully Paid", "Charged Off", "Default"]
    )

    x["usable_36m"] = (
        x["terminal"] |
        (x["followup_months"] >= 36)
    )

    x["default"] = x["loan_status"].isin(
        ["Charged Off", "Default"]
    ).astype(int)

    parts.append(x)

    print(f"Finished chunk {i}")

audit_2015_fixed = pd.concat(parts, ignore_index=True)

summary_2015_fixed = (
    audit_2015_fixed
    .groupby("split")
    .agg(
        n=("default", "size"),
        usable_36m=("usable_36m", "sum"),
        defaults=("default", "sum"),
        default_rate=("default", "mean")
    )
)

summary_2015_fixed["incomplete"] = (
    summary_2015_fixed["n"] -
    summary_2015_fixed["usable_36m"]
)

print("\n2015-ONLY FIXED-HORIZON DESIGN")
print(summary_2015_fixed)

print("\nINCOMPLETE RECORDS")
print(
    audit_2015_fixed.loc[
        ~audit_2015_fixed["usable_36m"],
        ["split", "issue_date", "loan_status",
         "last_date", "followup_months"]
    ].to_string(index=False)
)

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

2015-ONLY FIXED-HORIZON DESIGN
                 n  usable_36m  defaults  default_rate  incomplete
split                                                             
Test         88784       88784     13172      0.148360           0
Train       120766      120766     18264      0.151235           0
Validation   54794       54794      7921      0.144560           0

INCOMPLETE RECORDS
Empty DataFrame
Columns: [split, issue_date, loan_status, last_date, followup_months]
Index: []


In [23]:
# ============================================================
# W1 - Step 19: Extract clean fixed-horizon modelling dataset
# ============================================================

import os
import pandas as pd

MODEL_COLS = (
    planned_features
    + [
        "issue_date",
        "loan_status",
        "last_date"
    ]
)

parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=MODEL_COLS,
    convert_categoricals=False,
    chunksize=250_000
)

for i, chunk in enumerate(reader, start=1):

    # 36-month loans only
    term_months = (
        chunk["term"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )

    chunk = chunk.loc[term_months == 36].copy()

    chunk["split"] = pd.NA

    # Train: Jan-Jun 2015
    chunk.loc[
        (chunk["issue_date"] >= "2015-01-01") &
        (chunk["issue_date"] <  "2015-07-01"),
        "split"
    ] = "Train"

    # Validation: Jul-Aug 2015
    chunk.loc[
        (chunk["issue_date"] >= "2015-07-01") &
        (chunk["issue_date"] <  "2015-09-01"),
        "split"
    ] = "Validation"

    # Test: Oct-Dec 2015
    chunk.loc[
        (chunk["issue_date"] >= "2015-10-01") &
        (chunk["issue_date"] <  "2016-01-01"),
        "split"
    ] = "Test"

    chunk = chunk.loc[chunk["split"].notna()].copy()

    # Fixed-horizon default outcome
    chunk["default"] = chunk["loan_status"].isin(
        ["Charged Off", "Default"]
    ).astype("int8")

    parts.append(chunk)

    print(f"Finished chunk {i}")

df_fixed36 = pd.concat(parts, ignore_index=True)

print("\nFINAL MODELLING DATASET SHAPE:")
print(df_fixed36.shape)

print("\nSPLIT COUNTS:")
print(df_fixed36["split"].value_counts())

print("\nDEFAULT RATE BY SPLIT:")
print(
    df_fixed36.groupby("split")["default"]
    .agg(["count", "sum", "mean"])
)

print("\nTOTAL MISSING VALUES IN 41 PREDICTORS:")
print(df_fixed36[planned_features].isna().sum().sum())

print("\nTOP 15 FEATURES BY MISSING COUNT:")
print(
    df_fixed36[planned_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

FINAL MODELLING DATASET SHAPE:
(264344, 46)

SPLIT COUNTS:
split
Train         120766
Test           88784
Validation     54794
Name: count, dtype: int64

DEFAULT RATE BY SPLIT:
             count    sum      mean
split                              
Test         88784  13172  0.148360
Train       120766  18264  0.151235
Validation   54794   7921  0.144560

TOTAL MISSING VALUES IN 41 PREDICTORS:
115

TOP 15 FEATURES BY MISSING COUNT:
revol_util              114
num_rev_accts             1
emp_length                0
dti                       0
verification_status       0
delinq_2yrs               0
pub_rec                   0
home_ownership            0
annual_inc                0
tax_liens                 0
mort_acc                  0
pub_rec_bankruptcies      0
loan_amnt                 0
int_rate 

In [24]:
# ============================================================
# W1 - Step 20: Identify numeric/categorical predictors
# and inspect categorical cardinality
# ============================================================

# Split feature frame
X_train = df_fixed36.loc[
    df_fixed36["split"] == "Train",
    planned_features
].copy()

X_val = df_fixed36.loc[
    df_fixed36["split"] == "Validation",
    planned_features
].copy()

X_test = df_fixed36.loc[
    df_fixed36["split"] == "Test",
    planned_features
].copy()

y_train = df_fixed36.loc[
    df_fixed36["split"] == "Train",
    "default"
].astype(int).to_numpy()

y_val = df_fixed36.loc[
    df_fixed36["split"] == "Validation",
    "default"
].astype(int).to_numpy()

y_test = df_fixed36.loc[
    df_fixed36["split"] == "Test",
    "default"
].astype(int).to_numpy()


# Detect data types from TRAIN only
cat_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

num_cols = [
    c for c in planned_features
    if c not in cat_cols
]


print("TRAIN:", X_train.shape, "Default rate:", y_train.mean())
print("VAL  :", X_val.shape,   "Default rate:", y_val.mean())
print("TEST :", X_test.shape,  "Default rate:", y_test.mean())

print("\nNumeric predictors:", len(num_cols))
print(num_cols)

print("\nCategorical predictors:", len(cat_cols))
print(cat_cols)

print("\nCATEGORICAL CARDINALITY — TRAIN ONLY")
for c in cat_cols:
    print(
        f"{c:25s}",
        "levels =", X_train[c].nunique(dropna=False)
    )

TRAIN: (120766, 41) Default rate: 0.15123461901528576
VAL  : (54794, 41) Default rate: 0.1445596233164215
TEST : (88784, 41) Default rate: 0.14836006487655434

Numeric predictors: 33
['annual_inc', 'dti', 'delinq_2yrs', 'pub_rec', 'mort_acc', 'tax_liens', 'pub_rec_bankruptcies', 'loan_amnt', 'funded_amnt', 'int_rate', 'installment', 'open_acc', 'total_acc', 'revol_bal', 'revol_util', 'pct_tl_nvr_dlq', 'num_rev_accts', 'num_sats', 'tot_hi_cred_lim', 'total_bc_limit', 'unemployment', 'unem_rate', 'gdpindex', 'gdpreal', 'gdpcontrib', 'cpi', 'inf', 'FEDFUNDS', 'gdppercap', 'riskprem', 'issue_year', 'quarter', 'month']

Categorical predictors: 8
['emp_length', 'home_ownership', 'verification_status', 'term', 'grade', 'sub_grade', 'purpose', 'initial_list_status']

CATEGORICAL CARDINALITY — TRAIN ONLY
emp_length                levels = 12
home_ownership            levels = 3
verification_status       levels = 3
term                      levels = 1
grade                     levels = 7
sub_gra

In [25]:
# ============================================================
# W1 - Step 21: Check constant predictors and unseen categories
# ============================================================

# Number of unique values in training data
train_nunique = X_train.nunique(dropna=False)

constant_cols = train_nunique[
    train_nunique <= 1
].index.tolist()

print("CONSTANT / ZERO-VARIANCE TRAINING FEATURES:")
for c in constant_cols:
    print(
        f"{c:25s}",
        "unique =", train_nunique[c],
        "| value =", X_train[c].drop_duplicates().tolist()
    )

print("\nUNSEEN CATEGORICAL LEVELS IN VALIDATION / TEST:")

for c in cat_cols:
    train_levels = set(
        X_train[c].dropna().astype(str).unique()
    )
    val_levels = set(
        X_val[c].dropna().astype(str).unique()
    )
    test_levels = set(
        X_test[c].dropna().astype(str).unique()
    )

    unseen_val = sorted(val_levels - train_levels)
    unseen_test = sorted(test_levels - train_levels)

    print(f"\n{c}")
    print("  Train levels :", len(train_levels))
    print("  Unseen in val :", unseen_val)
    print("  Unseen in test:", unseen_test)

CONSTANT / ZERO-VARIANCE TRAINING FEATURES:
term                      unique = 1 | value = [' 36 months']
issue_year                unique = 1 | value = [2015.0]

UNSEEN CATEGORICAL LEVELS IN VALIDATION / TEST:

emp_length
  Train levels : 12
  Unseen in val : []
  Unseen in test: []

home_ownership
  Train levels : 3
  Unseen in val : []
  Unseen in test: ['ANY']

verification_status
  Train levels : 3
  Unseen in val : []
  Unseen in test: []

term
  Train levels : 1
  Unseen in val : []
  Unseen in test: []

grade
  Train levels : 7
  Unseen in val : []
  Unseen in test: []

sub_grade
  Train levels : 35
  Unseen in val : []
  Unseen in test: []

purpose
  Train levels : 13
  Unseen in val : []
  Unseen in test: []

initial_list_status
  Train levels : 2
  Unseen in val : []
  Unseen in test: []


In [26]:
# ============================================================
# W1 - Step 22: Build and fit training-only preprocessing
# ============================================================

import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


# Remove zero-variance predictors
model_features = [
    c for c in planned_features
    if c not in ["term", "issue_year"]
]

model_cat_cols = [
    c for c in cat_cols
    if c in model_features
]

model_num_cols = [
    c for c in num_cols
    if c in model_features
]


# Rebuild matrices with final feature list
X_train_model = X_train[model_features].copy()
X_val_model   = X_val[model_features].copy()
X_test_model  = X_test[model_features].copy()


# Numeric pipeline
numeric_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


# Categorical pipeline
categorical_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


# Combined preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipe,
            model_num_cols
        ),
        (
            "categorical",
            categorical_pipe,
            model_cat_cols
        )
    ],
    remainder="drop"
)


# Fit ONLY on training data
X_train_proc = preprocessor.fit_transform(
    X_train_model
)

# Frozen transformation for val/test
X_val_proc = preprocessor.transform(
    X_val_model
)

X_test_proc = preprocessor.transform(
    X_test_model
)


# Feature names after preprocessing
feature_names = (
    preprocessor
    .get_feature_names_out()
)


print("Raw model predictors:", len(model_features))
print("Numeric predictors:", len(model_num_cols))
print("Categorical predictors:", len(model_cat_cols))

print("\nTRANSFORMED SHAPES")
print("Train:", X_train_proc.shape)
print("Val  :", X_val_proc.shape)
print("Test :", X_test_proc.shape)

print("\nFinal encoded feature count:",
      len(feature_names))

print("\nAny NaN after preprocessing?")
print("Train:", np.isnan(X_train_proc).any())
print("Val  :", np.isnan(X_val_proc).any())
print("Test :", np.isnan(X_test_proc).any())

print("\nAny infinite values?")
print("Train:", np.isinf(X_train_proc).any())
print("Val  :", np.isinf(X_val_proc).any())
print("Test :", np.isinf(X_test_proc).any())

print("\nFirst 20 encoded feature names:")
for f in feature_names[:20]:
    print(f)

Raw model predictors: 39
Numeric predictors: 32
Categorical predictors: 7

TRANSFORMED SHAPES
Train: (120766, 107)
Val  : (54794, 107)
Test : (88784, 107)

Final encoded feature count: 107

Any NaN after preprocessing?
Train: False
Val  : False
Test : False

Any infinite values?
Train: False
Val  : False
Test : False

First 20 encoded feature names:
numeric__annual_inc
numeric__dti
numeric__delinq_2yrs
numeric__pub_rec
numeric__mort_acc
numeric__tax_liens
numeric__pub_rec_bankruptcies
numeric__loan_amnt
numeric__funded_amnt
numeric__int_rate
numeric__installment
numeric__open_acc
numeric__total_acc
numeric__revol_bal
numeric__revol_util
numeric__pct_tl_nvr_dlq
numeric__num_rev_accts
numeric__num_sats
numeric__tot_hi_cred_lim
numeric__total_bc_limit


In [27]:
# ============================================================
# W1 - Step 23: Train SGD logistic baseline across 3 seeds
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)

SEEDS = [17, 42, 2026]

sgd_results = []
sgd_predictions = {}

for seed in SEEDS:

    print("\n" + "=" * 70)
    print(f"SGD LOGISTIC - SEED {seed}")
    print("=" * 70)

    model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-5,
        max_iter=2000,
        tol=1e-4,
        random_state=seed
    )

    model.fit(X_train_proc, y_train)

    p_val = model.predict_proba(X_val_proc)[:, 1]
    p_test = model.predict_proba(X_test_proc)[:, 1]

    # Numerical safety for log loss
    p_val = np.clip(p_val, 1e-7, 1 - 1e-7)
    p_test = np.clip(p_test, 1e-7, 1 - 1e-7)

    sgd_predictions[seed] = {
        "val": p_val.copy(),
        "test": p_test.copy()
    }

    row = {
        "seed": seed,

        "val_auc": roc_auc_score(y_val, p_val),
        "test_auc": roc_auc_score(y_test, p_test),

        "val_ap": average_precision_score(y_val, p_val),
        "test_ap": average_precision_score(y_test, p_test),

        "val_brier": brier_score_loss(y_val, p_val),
        "test_brier": brier_score_loss(y_test, p_test),

        "val_logloss": log_loss(y_val, p_val),
        "test_logloss": log_loss(y_test, p_test),

        "n_iter": int(model.n_iter_)
    }

    sgd_results.append(row)

    print("Iterations :", model.n_iter_)
    print(f"Val AUC    : {row['val_auc']:.6f}")
    print(f"Test AUC   : {row['test_auc']:.6f}")
    print(f"Val AP     : {row['val_ap']:.6f}")
    print(f"Test AP    : {row['test_ap']:.6f}")
    print(f"Val Brier  : {row['val_brier']:.6f}")
    print(f"Test Brier : {row['test_brier']:.6f}")
    print(f"Val LogLoss: {row['val_logloss']:.6f}")
    print(f"Test LogLoss:{row['test_logloss']:.6f}")


sgd_results_df = pd.DataFrame(sgd_results)

print("\n" + "=" * 70)
print("SGD LOGISTIC - THREE-SEED SUMMARY")
print("=" * 70)

print(sgd_results_df.to_string(index=False))

print("\nMEAN ± SD")
for metric in [
    "val_auc", "test_auc",
    "val_ap", "test_ap",
    "val_brier", "test_brier",
    "val_logloss", "test_logloss"
]:
    print(
        f"{metric:15s}: "
        f"{sgd_results_df[metric].mean():.6f} "
        f"± {sgd_results_df[metric].std(ddof=1):.6f}"
    )


SGD LOGISTIC - SEED 17
Iterations : 98
Val AUC    : 0.776167
Test AUC   : 0.737776
Val AP     : 0.353609
Test AP    : 0.318833
Val Brier  : 0.108838
Test Brier : 0.117105
Val LogLoss: 0.369520
Test LogLoss:0.418802

SGD LOGISTIC - SEED 42
Iterations : 82
Val AUC    : 0.780995
Test AUC   : 0.742453
Val AP     : 0.348925
Test AP    : 0.320024
Val Brier  : 0.108502
Test Brier : 0.115839
Val LogLoss: 0.353763
Test LogLoss:0.389149

SGD LOGISTIC - SEED 2026
Iterations : 102
Val AUC    : 0.785743
Test AUC   : 0.743581
Val AP     : 0.362191
Test AP    : 0.326796
Val Brier  : 0.106849
Test Brier : 0.114833
Val LogLoss: 0.350790
Test LogLoss:0.390726

SGD LOGISTIC - THREE-SEED SUMMARY
 seed  val_auc  test_auc   val_ap  test_ap  val_brier  test_brier  val_logloss  test_logloss  n_iter
   17 0.776167  0.737776 0.353609 0.318833   0.108838    0.117105     0.369520      0.418802      98
   42 0.780995  0.742453 0.348925 0.320024   0.108502    0.115839     0.353763      0.389149      82
 2026 0.785

In [28]:
# ============================================================
# W1 - Step 24: Train HistGradientBoosting across 3 seeds
# Full corrected fixed-horizon training cohort
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)

SEEDS = [17, 42, 2026]

histgb_results = []
histgb_predictions = {}

for seed in SEEDS:

    print("\n" + "=" * 70)
    print(f"HISTGRADIENTBOOSTING - SEED {seed}")
    print("=" * 70)

    model = HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        max_iter=300,
        max_depth=6,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=20,
        random_state=seed
    )

    model.fit(X_train_proc, y_train)

    p_val = model.predict_proba(X_val_proc)[:, 1]
    p_test = model.predict_proba(X_test_proc)[:, 1]

    p_val = np.clip(p_val, 1e-7, 1 - 1e-7)
    p_test = np.clip(p_test, 1e-7, 1 - 1e-7)

    histgb_predictions[seed] = {
        "val": p_val.copy(),
        "test": p_test.copy()
    }

    row = {
        "seed": seed,

        "val_auc": roc_auc_score(y_val, p_val),
        "test_auc": roc_auc_score(y_test, p_test),

        "val_ap": average_precision_score(y_val, p_val),
        "test_ap": average_precision_score(y_test, p_test),

        "val_brier": brier_score_loss(y_val, p_val),
        "test_brier": brier_score_loss(y_test, p_test),

        "val_logloss": log_loss(y_val, p_val),
        "test_logloss": log_loss(y_test, p_test),

        "n_iter": int(model.n_iter_)
    }

    histgb_results.append(row)

    print("Iterations  :", model.n_iter_)
    print(f"Val AUC     : {row['val_auc']:.6f}")
    print(f"Test AUC    : {row['test_auc']:.6f}")
    print(f"Val AP      : {row['val_ap']:.6f}")
    print(f"Test AP     : {row['test_ap']:.6f}")
    print(f"Val Brier   : {row['val_brier']:.6f}")
    print(f"Test Brier  : {row['test_brier']:.6f}")
    print(f"Val LogLoss : {row['val_logloss']:.6f}")
    print(f"Test LogLoss: {row['test_logloss']:.6f}")


histgb_results_df = pd.DataFrame(histgb_results)

print("\n" + "=" * 70)
print("HISTGRADIENTBOOSTING - THREE-SEED SUMMARY")
print("=" * 70)

print(histgb_results_df.to_string(index=False))

print("\nMEAN ± SD")
for metric in [
    "val_auc", "test_auc",
    "val_ap", "test_ap",
    "val_brier", "test_brier",
    "val_logloss", "test_logloss"
]:
    print(
        f"{metric:15s}: "
        f"{histgb_results_df[metric].mean():.6f} "
        f"± {histgb_results_df[metric].std(ddof=1):.6f}"
    )


HISTGRADIENTBOOSTING - SEED 17
Iterations  : 260
Val AUC     : 0.792621
Test AUC    : 0.751417
Val AP      : 0.400018
Test AP     : 0.379394
Val Brier   : 0.103924
Test Brier  : 0.110282
Val LogLoss : 0.343738
Test LogLoss: 0.376117

HISTGRADIENTBOOSTING - SEED 42
Iterations  : 212
Val AUC     : 0.792740
Test AUC    : 0.754081
Val AP      : 0.398103
Test AP     : 0.379573
Val Brier   : 0.104236
Test Brier  : 0.110456
Val LogLoss : 0.344250
Test LogLoss: 0.376587

HISTGRADIENTBOOSTING - SEED 2026
Iterations  : 164
Val AUC     : 0.790397
Test AUC    : 0.749633
Val AP      : 0.398299
Test AP     : 0.379524
Val Brier   : 0.104290
Test Brier  : 0.110474
Val LogLoss : 0.344841
Test LogLoss: 0.376137

HISTGRADIENTBOOSTING - THREE-SEED SUMMARY
 seed  val_auc  test_auc   val_ap  test_ap  val_brier  test_brier  val_logloss  test_logloss  n_iter
   17 0.792621  0.751417 0.400018 0.379394   0.103924    0.110282     0.343738      0.376117     260
   42 0.792740  0.754081 0.398103 0.379573   0.1042

In [29]:
# ============================================================
# W1 - Step 25: Train MLP across 3 seeds
# Same corrected fixed-horizon cohort
# ============================================================

import numpy as np
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)

SEEDS = [17, 42, 2026]

mlp_results = []
mlp_predictions = {}

for seed in SEEDS:

    print("\n" + "=" * 70)
    print(f"MLP - SEED {seed}")
    print("=" * 70)

    model = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=512,
        learning_rate_init=0.001,
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=10,
        random_state=seed
    )

    model.fit(X_train_proc, y_train)

    p_val = model.predict_proba(X_val_proc)[:, 1]
    p_test = model.predict_proba(X_test_proc)[:, 1]

    p_val = np.clip(p_val, 1e-7, 1 - 1e-7)
    p_test = np.clip(p_test, 1e-7, 1 - 1e-7)

    mlp_predictions[seed] = {
        "val": p_val.copy(),
        "test": p_test.copy()
    }

    row = {
        "seed": seed,

        "val_auc": roc_auc_score(y_val, p_val),
        "test_auc": roc_auc_score(y_test, p_test),

        "val_ap": average_precision_score(y_val, p_val),
        "test_ap": average_precision_score(y_test, p_test),

        "val_brier": brier_score_loss(y_val, p_val),
        "test_brier": brier_score_loss(y_test, p_test),

        "val_logloss": log_loss(y_val, p_val),
        "test_logloss": log_loss(y_test, p_test),

        "n_iter": int(model.n_iter_)
    }

    mlp_results.append(row)

    print("Iterations  :", model.n_iter_)
    print(f"Val AUC     : {row['val_auc']:.6f}")
    print(f"Test AUC    : {row['test_auc']:.6f}")
    print(f"Val AP      : {row['val_ap']:.6f}")
    print(f"Test AP     : {row['test_ap']:.6f}")
    print(f"Val Brier   : {row['val_brier']:.6f}")
    print(f"Test Brier  : {row['test_brier']:.6f}")
    print(f"Val LogLoss : {row['val_logloss']:.6f}")
    print(f"Test LogLoss: {row['test_logloss']:.6f}")


mlp_results_df = pd.DataFrame(mlp_results)

print("\n" + "=" * 70)
print("MLP - THREE-SEED SUMMARY")
print("=" * 70)

print(mlp_results_df.to_string(index=False))

print("\nMEAN ± SD")
for metric in [
    "val_auc", "test_auc",
    "val_ap", "test_ap",
    "val_brier", "test_brier",
    "val_logloss", "test_logloss"
]:
    print(
        f"{metric:15s}: "
        f"{mlp_results_df[metric].mean():.6f} "
        f"± {mlp_results_df[metric].std(ddof=1):.6f}"
    )


MLP - SEED 17
Iterations  : 20
Val AUC     : 0.799898
Test AUC    : 0.771452
Val AP      : 0.386263
Test AP     : 0.377125
Val Brier   : 0.105160
Test Brier  : 0.111292
Val LogLoss : 0.346314
Test LogLoss: 0.385625

MLP - SEED 42
Iterations  : 13
Val AUC     : 0.798872
Test AUC    : 0.766150
Val AP      : 0.389892
Test AP     : 0.379262
Val Brier   : 0.104609
Test Brier  : 0.110626
Val LogLoss : 0.342550
Test LogLoss: 0.376539

MLP - SEED 2026
Iterations  : 14
Val AUC     : 0.801246
Test AUC    : 0.771441
Val AP      : 0.390862
Test AP     : 0.380281
Val Brier   : 0.104661
Test Brier  : 0.110825
Val LogLoss : 0.343447
Test LogLoss: 0.381051

MLP - THREE-SEED SUMMARY
 seed  val_auc  test_auc   val_ap  test_ap  val_brier  test_brier  val_logloss  test_logloss  n_iter
   17 0.799898  0.771452 0.386263 0.377125   0.105160    0.111292     0.346314      0.385625      20
   42 0.798872  0.766150 0.389892 0.379262   0.104609    0.110626     0.342550      0.376539      13
 2026 0.801246  0.771

In [30]:
# ============================================================
# W3 - Step 26: Paired DeLong AUC comparisons
# Same test observations, matched seeds
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import norm


def compute_midrank(x):
    """Compute midranks for tied observations."""
    x = np.asarray(x)
    order = np.argsort(x)
    sorted_x = x[order]

    n = len(x)
    ranks = np.zeros(n, dtype=float)

    i = 0
    while i < n:
        j = i
        while j < n and sorted_x[j] == sorted_x[i]:
            j += 1

        # +1 because DeLong formulation uses ranks starting at 1
        midrank = 0.5 * (i + j - 1) + 1
        ranks[i:j] = midrank
        i = j

    out = np.empty(n, dtype=float)
    out[order] = ranks

    return out


def fast_delong(predictions_sorted, n_positive):
    """
    Fast DeLong covariance calculation.

    predictions_sorted:
        shape = (n_models, n_samples)
        samples ordered with positives first, negatives second
    """
    m = n_positive
    n = predictions_sorted.shape[1] - m
    k = predictions_sorted.shape[0]

    positive = predictions_sorted[:, :m]
    negative = predictions_sorted[:, m:]

    tx = np.empty((k, m))
    ty = np.empty((k, n))
    tz = np.empty((k, m + n))

    for r in range(k):
        tx[r] = compute_midrank(positive[r])
        ty[r] = compute_midrank(negative[r])
        tz[r] = compute_midrank(predictions_sorted[r])

    aucs = (
        tz[:, :m].sum(axis=1) / (m * n)
        - (m + 1.0) / (2.0 * n)
    )

    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m

    sx = np.cov(v01)
    sy = np.cov(v10)

    covariance = sx / m + sy / n

    return aucs, covariance


def paired_delong(y_true, pred_a, pred_b):
    """
    Compare AUC(A) - AUC(B) using paired DeLong.
    """
    y_true = np.asarray(y_true).astype(int)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    # positives first, negatives second
    order = np.argsort(-y_true)

    y_sorted = y_true[order]

    predictions_sorted = np.vstack([
        pred_a[order],
        pred_b[order]
    ])

    n_positive = int(y_sorted.sum())

    aucs, covariance = fast_delong(
        predictions_sorted,
        n_positive
    )

    diff = aucs[0] - aucs[1]

    variance = (
        covariance[0, 0]
        + covariance[1, 1]
        - 2 * covariance[0, 1]
    )

    variance = max(float(variance), 0.0)
    se = np.sqrt(variance)

    if se > 0:
        z = diff / se
        p = 2 * norm.sf(abs(z))

        ci_low = diff - 1.96 * se
        ci_high = diff + 1.96 * se
    else:
        z = np.nan
        p = np.nan
        ci_low = diff
        ci_high = diff

    return {
        "auc_a": aucs[0],
        "auc_b": aucs[1],
        "delta_auc": diff,
        "se": se,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "z": z,
        "p_value": p
    }


comparison_rows = []

comparisons = [
    ("MLP", mlp_predictions,
     "HistGB", histgb_predictions),

    ("MLP", mlp_predictions,
     "SGD Logistic", sgd_predictions),

    ("HistGB", histgb_predictions,
     "SGD Logistic", sgd_predictions)
]


for seed in SEEDS:

    print("\n" + "=" * 75)
    print(f"PAIRED DELONG TESTS - SEED {seed}")
    print("=" * 75)

    for name_a, preds_a, name_b, preds_b in comparisons:

        result = paired_delong(
            y_test,
            preds_a[seed]["test"],
            preds_b[seed]["test"]
        )

        comparison_rows.append({
            "seed": seed,
            "model_a": name_a,
            "model_b": name_b,
            **result
        })

        print(f"\n{name_a} vs {name_b}")
        print(
            f"AUCs       : "
            f"{result['auc_a']:.6f} vs "
            f"{result['auc_b']:.6f}"
        )
        print(
            f"Delta AUC  : "
            f"{result['delta_auc']:.6f}"
        )
        print(
            f"95% CI     : "
            f"[{result['ci_low']:.6f}, "
            f"{result['ci_high']:.6f}]"
        )
        print(
            f"p-value    : "
            f"{result['p_value']:.6g}"
        )


delong_results_df = pd.DataFrame(comparison_rows)


print("\n" + "=" * 75)
print("PAIRED DELONG SUMMARY")
print("=" * 75)

print(
    delong_results_df[
        [
            "seed",
            "model_a",
            "model_b",
            "auc_a",
            "auc_b",
            "delta_auc",
            "ci_low",
            "ci_high",
            "p_value"
        ]
    ].to_string(index=False)
)


print("\nMEAN DELTA AUC ACROSS THE THREE MATCHED SEEDS")

print(
    delong_results_df
    .groupby(["model_a", "model_b"])["delta_auc"]
    .agg(["mean", "std", "min", "max"])
)


PAIRED DELONG TESTS - SEED 17

MLP vs HistGB
AUCs       : 0.771452 vs 0.751417
Delta AUC  : 0.020035
95% CI     : [0.017874, 0.022196]
p-value    : 8.2224e-74

MLP vs SGD Logistic
AUCs       : 0.771452 vs 0.737776
Delta AUC  : 0.033676
95% CI     : [0.030891, 0.036461]
p-value    : 3.64282e-124

HistGB vs SGD Logistic
AUCs       : 0.751417 vs 0.737776
Delta AUC  : 0.013641
95% CI     : [0.010378, 0.016905]
p-value    : 2.56585e-16

PAIRED DELONG TESTS - SEED 42

MLP vs HistGB
AUCs       : 0.766150 vs 0.754081
Delta AUC  : 0.012070
95% CI     : [0.010267, 0.013872]
p-value    : 2.32085e-39

MLP vs SGD Logistic
AUCs       : 0.766150 vs 0.742453
Delta AUC  : 0.023697
95% CI     : [0.021748, 0.025647]
p-value    : 1.99492e-125

HistGB vs SGD Logistic
AUCs       : 0.754081 vs 0.742453
Delta AUC  : 0.011628
95% CI     : [0.009040, 0.014216]
p-value    : 1.2959e-18

PAIRED DELONG TESTS - SEED 2026

MLP vs HistGB
AUCs       : 0.771441 vs 0.749633
Delta AUC  : 0.021807
95% CI     : [0.019752, 

In [31]:
# ============================================================
# W2 - Step 27: Brier Skill Score and log-loss skill
# against the constant base-rate predictor
# BEFORE recalibration
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss


def base_rate_scores(y):
    """
    Constant predictor equal to observed prevalence
    in the evaluation cohort.
    """
    y = np.asarray(y).astype(int)

    prevalence = y.mean()

    p0 = np.full(
        len(y),
        prevalence,
        dtype=float
    )

    brier0 = brier_score_loss(y, p0)
    logloss0 = log_loss(y, p0)

    return prevalence, brier0, logloss0


# Evaluation-cohort reference scores
val_prev, val_brier0, val_logloss0 = base_rate_scores(y_val)
test_prev, test_brier0, test_logloss0 = base_rate_scores(y_test)


print("=" * 75)
print("CONSTANT BASE-RATE REFERENCES")
print("=" * 75)

print(f"Validation prevalence : {val_prev:.6f}")
print(f"Validation Brier_0    : {val_brier0:.6f}")
print(f"Validation LogLoss_0  : {val_logloss0:.6f}")

print()

print(f"Test prevalence       : {test_prev:.6f}")
print(f"Test Brier_0          : {test_brier0:.6f}")
print(f"Test LogLoss_0        : {test_logloss0:.6f}")


models_predictions = {
    "SGD Logistic": sgd_predictions,
    "HistGB": histgb_predictions,
    "MLP": mlp_predictions
}


skill_rows = []

for model_name, pred_dict in models_predictions.items():

    for seed in SEEDS:

        p_val = pred_dict[seed]["val"]
        p_test = pred_dict[seed]["test"]

        val_brier = brier_score_loss(
            y_val,
            p_val
        )

        test_brier = brier_score_loss(
            y_test,
            p_test
        )

        val_ll = log_loss(
            y_val,
            p_val
        )

        test_ll = log_loss(
            y_test,
            p_test
        )

        # Skill > 0 means better than constant base-rate predictor
        val_bss = 1 - (val_brier / val_brier0)
        test_bss = 1 - (test_brier / test_brier0)

        val_ll_skill = 1 - (val_ll / val_logloss0)
        test_ll_skill = 1 - (test_ll / test_logloss0)

        skill_rows.append({
            "model": model_name,
            "seed": seed,

            "val_brier": val_brier,
            "val_brier_skill": val_bss,
            "val_logloss": val_ll,
            "val_logloss_skill": val_ll_skill,

            "test_brier": test_brier,
            "test_brier_skill": test_bss,
            "test_logloss": test_ll,
            "test_logloss_skill": test_ll_skill
        })


skill_df = pd.DataFrame(skill_rows)


print("\n" + "=" * 75)
print("PER-SEED SKILL SCORES")
print("=" * 75)

print(
    skill_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n" + "=" * 75)
print("THREE-SEED MEAN ± SD")
print("=" * 75)

for model_name in skill_df["model"].unique():

    x = skill_df[
        skill_df["model"] == model_name
    ]

    print(f"\n{model_name}")

    for metric in [
        "test_brier",
        "test_brier_skill",
        "test_logloss",
        "test_logloss_skill"
    ]:

        print(
            f"{metric:20s}: "
            f"{x[metric].mean():.6f} "
            f"± {x[metric].std(ddof=1):.6f}"
        )

CONSTANT BASE-RATE REFERENCES
Validation prevalence : 0.144560
Validation Brier_0    : 0.123662
Validation LogLoss_0  : 0.413155

Test prevalence       : 0.148360
Test Brier_0          : 0.126349
Test LogLoss_0        : 0.419854

PER-SEED SKILL SCORES
       model  seed  val_brier  val_brier_skill  val_logloss  val_logloss_skill  test_brier  test_brier_skill  test_logloss  test_logloss_skill
SGD Logistic    17   0.108838         0.119876     0.369520           0.105613    0.117105          0.073164      0.418802            0.002505
SGD Logistic    42   0.108502         0.122590     0.353763           0.143752    0.115839          0.083185      0.389149            0.073132
SGD Logistic  2026   0.106849         0.135963     0.350790           0.150949    0.114833          0.091147      0.390726            0.069377
      HistGB    17   0.103924         0.159617     0.343738           0.168016    0.110282          0.127166      0.376117            0.104172
      HistGB    42   0.104236    

In [32]:
# ============================================================
# W2 - Step 28: Validation-fitted logistic recalibration
# Evaluate recalibrated probabilities on TEST only
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
    average_precision_score
)


def logit_clip(p, eps=1e-6):
    p = np.clip(
        np.asarray(p, dtype=float),
        eps,
        1 - eps
    )
    return np.log(p / (1 - p))


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-z))


calibration_rows = []
calibrated_predictions = {}


for model_name, pred_dict in models_predictions.items():

    calibrated_predictions[model_name] = {}

    for seed in SEEDS:

        # ----------------------------------------------------
        # 1. Validation predictions are the ONLY data used
        #    to fit recalibration parameters
        # ----------------------------------------------------
        p_val_raw = pred_dict[seed]["val"]
        p_test_raw = pred_dict[seed]["test"]

        z_val = logit_clip(p_val_raw).reshape(-1, 1)
        z_test = logit_clip(p_test_raw).reshape(-1, 1)

        # Logistic recalibration:
        # logit(p_cal) = intercept + slope * logit(p_raw)
        calibrator = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=1000
        )

        calibrator.fit(
            z_val,
            y_val
        )

        intercept = float(
            calibrator.intercept_[0]
        )

        slope = float(
            calibrator.coef_[0, 0]
        )

        # ----------------------------------------------------
        # 2. Apply frozen validation-fitted calibration
        #    to TEST predictions
        # ----------------------------------------------------
        p_test_cal = calibrator.predict_proba(
            z_test
        )[:, 1]

        p_test_cal = np.clip(
            p_test_cal,
            1e-7,
            1 - 1e-7
        )

        calibrated_predictions[model_name][seed] = (
            p_test_cal.copy()
        )

        # ----------------------------------------------------
        # 3. Test metrics before and after calibration
        # ----------------------------------------------------
        raw_brier = brier_score_loss(
            y_test,
            p_test_raw
        )

        cal_brier = brier_score_loss(
            y_test,
            p_test_cal
        )

        raw_ll = log_loss(
            y_test,
            p_test_raw
        )

        cal_ll = log_loss(
            y_test,
            p_test_cal
        )

        raw_bss = 1 - (
            raw_brier / test_brier0
        )

        cal_bss = 1 - (
            cal_brier / test_brier0
        )

        raw_ll_skill = 1 - (
            raw_ll / test_logloss0
        )

        cal_ll_skill = 1 - (
            cal_ll / test_logloss0
        )

        row = {
            "model": model_name,
            "seed": seed,

            "cal_intercept": intercept,
            "cal_slope": slope,

            "raw_test_auc":
                roc_auc_score(
                    y_test,
                    p_test_raw
                ),

            "cal_test_auc":
                roc_auc_score(
                    y_test,
                    p_test_cal
                ),

            "raw_test_ap":
                average_precision_score(
                    y_test,
                    p_test_raw
                ),

            "cal_test_ap":
                average_precision_score(
                    y_test,
                    p_test_cal
                ),

            "raw_test_brier":
                raw_brier,

            "cal_test_brier":
                cal_brier,

            "raw_brier_skill":
                raw_bss,

            "cal_brier_skill":
                cal_bss,

            "raw_test_logloss":
                raw_ll,

            "cal_test_logloss":
                cal_ll,

            "raw_logloss_skill":
                raw_ll_skill,

            "cal_logloss_skill":
                cal_ll_skill
        }

        calibration_rows.append(row)


calibration_df = pd.DataFrame(
    calibration_rows
)


print("=" * 80)
print("VALIDATION-FITTED RECALIBRATION — PER SEED")
print("=" * 80)

print(
    calibration_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n" + "=" * 80)
print("THREE-SEED MEAN ± SD — TEST")
print("=" * 80)

summary_metrics = [
    "cal_intercept",
    "cal_slope",
    "raw_test_brier",
    "cal_test_brier",
    "raw_brier_skill",
    "cal_brier_skill",
    "raw_test_logloss",
    "cal_test_logloss",
    "raw_logloss_skill",
    "cal_logloss_skill"
]

for model_name in calibration_df["model"].unique():

    x = calibration_df[
        calibration_df["model"] == model_name
    ]

    print(f"\n{model_name}")

    for metric in summary_metrics:

        print(
            f"{metric:22s}: "
            f"{x[metric].mean():.6f} "
            f"± {x[metric].std(ddof=1):.6f}"
        )

C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:

VALIDATION-FITTED RECALIBRATION — PER SEED
       model  seed  cal_intercept  cal_slope  raw_test_auc  cal_test_auc  raw_test_ap  cal_test_ap  raw_test_brier  cal_test_brier  raw_brier_skill  cal_brier_skill  raw_test_logloss  cal_test_logloss  raw_logloss_skill  cal_logloss_skill
SGD Logistic    17      -0.383787   0.615066      0.737776      0.737776     0.318833     0.318831        0.117105        0.114481         0.073164         0.093929          0.418802          0.381801           0.002505           0.090635
SGD Logistic    42      -0.344797   0.735816      0.742453      0.742453     0.320024     0.320024        0.115839        0.114070         0.083185         0.097186          0.389149          0.375150           0.073132           0.106474
SGD Logistic  2026      -0.188434   0.769241      0.743581      0.743581     0.326796     0.326796        0.114833        0.113734         0.091147         0.099842          0.390726          0.375572           0.069377           0.105469
 

In [33]:
# ============================================================
# W2 - Step 29: TEST calibration diagnostics
# Raw vs validation-recalibrated probabilities
#
# Reports:
#   - calibration intercept
#   - calibration slope
#   - 10-bin equal-frequency ECE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


def calibration_intercept_slope(y, p, eps=1e-6):
    """
    Diagnostic calibration model fitted on the evaluation set:
        logit(y) = intercept + slope * logit(p)

    Used only for assessment, not for recalibrating predictions.
    """
    y = np.asarray(y).astype(int)

    p = np.clip(
        np.asarray(p, dtype=float),
        eps,
        1 - eps
    )

    z = np.log(
        p / (1 - p)
    ).reshape(-1, 1)

    model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=1000
    )

    model.fit(z, y)

    intercept = float(
        model.intercept_[0]
    )

    slope = float(
        model.coef_[0, 0]
    )

    return intercept, slope


def ece_quantile(y, p, n_bins=10):
    """
    Expected Calibration Error using equal-frequency bins.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    # Rank-based quantile assignment avoids problems
    # from repeated probability values.
    order = np.argsort(p)

    bins = np.array_split(
        order,
        n_bins
    )

    ece = 0.0
    bin_rows = []

    n = len(y)

    for b, idx in enumerate(bins, start=1):

        if len(idx) == 0:
            continue

        mean_pred = p[idx].mean()
        observed = y[idx].mean()

        weight = len(idx) / n

        abs_gap = abs(
            observed - mean_pred
        )

        ece += weight * abs_gap

        bin_rows.append({
            "bin": b,
            "n": len(idx),
            "mean_pred": mean_pred,
            "observed_rate": observed,
            "abs_gap": abs_gap
        })

    return ece, pd.DataFrame(bin_rows)


test_calibration_rows = []


for model_name, pred_dict in models_predictions.items():

    for seed in SEEDS:

        p_raw = pred_dict[seed]["test"]

        p_cal = calibrated_predictions[
            model_name
        ][seed]

        # Raw diagnostics
        raw_intercept, raw_slope = (
            calibration_intercept_slope(
                y_test,
                p_raw
            )
        )

        raw_ece, _ = ece_quantile(
            y_test,
            p_raw,
            n_bins=10
        )

        # Recalibrated diagnostics
        cal_intercept, cal_slope = (
            calibration_intercept_slope(
                y_test,
                p_cal
            )
        )

        cal_ece, _ = ece_quantile(
            y_test,
            p_cal,
            n_bins=10
        )

        test_calibration_rows.append({
            "model": model_name,
            "seed": seed,

            "raw_intercept":
                raw_intercept,

            "raw_slope":
                raw_slope,

            "raw_ece":
                raw_ece,

            "cal_intercept":
                cal_intercept,

            "cal_slope":
                cal_slope,

            "cal_ece":
                cal_ece
        })


test_calibration_df = pd.DataFrame(
    test_calibration_rows
)


print("=" * 80)
print("TEST CALIBRATION DIAGNOSTICS — PER SEED")
print("=" * 80)

print(
    test_calibration_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n" + "=" * 80)
print("TEST CALIBRATION — THREE-SEED MEAN ± SD")
print("=" * 80)

metrics = [
    "raw_intercept",
    "raw_slope",
    "raw_ece",
    "cal_intercept",
    "cal_slope",
    "cal_ece"
]

for model_name in test_calibration_df["model"].unique():

    x = test_calibration_df[
        test_calibration_df["model"]
        == model_name
    ]

    print(f"\n{model_name}")

    for metric in metrics:

        print(
            f"{metric:18s}: "
            f"{x[metric].mean():.6f} "
            f"± {x[metric].std(ddof=1):.6f}"
        )

C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:

TEST CALIBRATION DIAGNOSTICS — PER SEED
       model  seed  raw_intercept  raw_slope  raw_ece  cal_intercept  cal_slope  cal_ece
SGD Logistic    17      -0.546292   0.464237 0.047463      -0.259776   0.753338 0.013856
SGD Logistic    42      -0.486383   0.582514 0.036873      -0.212803   0.791988 0.014927
SGD Logistic  2026      -0.347797   0.599645 0.036035      -0.200568   0.779677 0.016482
      HistGB    17      -0.151481   0.733154 0.029313      -0.070050   0.842213 0.025020
      HistGB    42      -0.166807   0.716258 0.029386      -0.066752   0.840355 0.023628
      HistGB  2026      -0.145754   0.744032 0.028083      -0.069543   0.845589 0.025035
         MLP    17      -0.192138   0.609125 0.043031      -0.095451   0.814862 0.022065
         MLP    42      -0.112829   0.697561 0.037443      -0.061198   0.838845 0.021926
         MLP  2026      -0.153176   0.646194 0.041215      -0.094413   0.815361 0.024724

TEST CALIBRATION — THREE-SEED MEAN ± SD

SGD Logistic
raw_intercept  

C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:

In [34]:
# ============================================================
# W4 - Step 30: Cost-sensitive threshold analysis
#
# Uses validation-fitted recalibrated probabilities.
#
# For each model / seed / cost ratio r:
#   1. theoretical threshold t* = 1 / (1 + r)
#   2. validation-optimal empirical threshold
#   3. evaluate both thresholds on TEST
#
# Cost:
#   C = r * FN + 1 * FP
#
# Normalized cost:
#   observed cost /
#   best trivial-policy cost
#
# where trivial policies are:
#   approve all  -> cost = r * prevalence
#   reject all   -> cost = 1 - prevalence
# ============================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


COST_RATIOS = [5, 10, 20]


def evaluate_cost(y, p, threshold, r):
    """
    Positive prediction = predicted default / reject.
    FN = actual default but approved.
    FP = non-default but rejected.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    pred = (p >= threshold).astype(int)

    tp = np.sum((y == 1) & (pred == 1))
    fp = np.sum((y == 0) & (pred == 1))
    fn = np.sum((y == 1) & (pred == 0))
    tn = np.sum((y == 0) & (pred == 0))

    n = len(y)
    prevalence = y.mean()

    raw_cost = (r * fn + fp) / n

    approve_all_cost = r * prevalence
    reject_all_cost = 1 - prevalence

    trivial_best = min(
        approve_all_cost,
        reject_all_cost
    )

    normalized_cost = (
        raw_cost / trivial_best
        if trivial_best > 0
        else np.nan
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    fnr = 1 - sensitivity
    fpr = 1 - specificity

    rejection_rate = pred.mean()

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "raw_cost": raw_cost,
        "normalized_cost": normalized_cost,
        "fnr": fnr,
        "fpr": fpr,
        "rejection_rate": rejection_rate
    }


def exact_optimal_threshold(y, p, r):
    """
    Find the exact empirical threshold minimizing:
        r*FN + FP

    Searches probability breakpoints rather than
    using an arbitrary threshold grid.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    order = np.argsort(-p)

    p_sorted = p[order]
    y_sorted = y[order]

    total_pos = y.sum()

    cum_tp = np.cumsum(y_sorted)
    cum_fp = np.cumsum(1 - y_sorted)

    # Evaluate only after the final observation
    # at each distinct probability value.
    distinct_end = np.r_[
        p_sorted[:-1] != p_sorted[1:],
        True
    ]

    candidate_idx = np.where(distinct_end)[0]

    tp = cum_tp[candidate_idx]
    fp = cum_fp[candidate_idx]
    fn = total_pos - tp

    costs = r * fn + fp

    # Also allow predicting nobody as default.
    none_cost = r * total_pos

    best_idx = np.argmin(costs)

    if none_cost < costs[best_idx]:
        return 1.0

    return float(
        p_sorted[candidate_idx[best_idx]]
    )


# ------------------------------------------------------------
# Reconstruct validation-fitted calibrators and retain
# calibrated validation + test predictions.
# ------------------------------------------------------------

cost_calibrated_predictions = {}

for model_name, pred_dict in models_predictions.items():

    cost_calibrated_predictions[model_name] = {}

    for seed in SEEDS:

        p_val_raw = pred_dict[seed]["val"]
        p_test_raw = pred_dict[seed]["test"]

        z_val = logit_clip(
            p_val_raw
        ).reshape(-1, 1)

        z_test = logit_clip(
            p_test_raw
        ).reshape(-1, 1)

        calibrator = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=1000
        )

        calibrator.fit(
            z_val,
            y_val
        )

        p_val_cal = calibrator.predict_proba(
            z_val
        )[:, 1]

        p_test_cal = calibrator.predict_proba(
            z_test
        )[:, 1]

        cost_calibrated_predictions[
            model_name
        ][seed] = {
            "val": p_val_cal,
            "test": p_test_cal
        }


# ------------------------------------------------------------
# Threshold analysis
# ------------------------------------------------------------

cost_rows = []

for model_name in models_predictions.keys():

    for seed in SEEDS:

        p_val_cal = (
            cost_calibrated_predictions[
                model_name
            ][seed]["val"]
        )

        p_test_cal = (
            cost_calibrated_predictions[
                model_name
            ][seed]["test"]
        )

        for r in COST_RATIOS:

            theoretical_t = 1 / (1 + r)

            val_opt_t = exact_optimal_threshold(
                y_val,
                p_val_cal,
                r
            )

            theoretical_test = evaluate_cost(
                y_test,
                p_test_cal,
                theoretical_t,
                r
            )

            valopt_test = evaluate_cost(
                y_test,
                p_test_cal,
                val_opt_t,
                r
            )

            cost_rows.append({
                "model": model_name,
                "seed": seed,
                "r": r,

                "theoretical_t":
                    theoretical_t,

                "val_opt_t":
                    val_opt_t,

                "theory_test_norm_cost":
                    theoretical_test[
                        "normalized_cost"
                    ],

                "valopt_test_norm_cost":
                    valopt_test[
                        "normalized_cost"
                    ],

                "theory_test_fnr":
                    theoretical_test["fnr"],

                "valopt_test_fnr":
                    valopt_test["fnr"],

                "theory_test_fpr":
                    theoretical_test["fpr"],

                "valopt_test_fpr":
                    valopt_test["fpr"],

                "theory_test_reject":
                    theoretical_test[
                        "rejection_rate"
                    ],

                "valopt_test_reject":
                    valopt_test[
                        "rejection_rate"
                    ]
            })


cost_threshold_df = pd.DataFrame(
    cost_rows
)


print("=" * 100)
print("W4 COST-SENSITIVE THRESHOLDS — PER SEED")
print("=" * 100)

print(
    cost_threshold_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n" + "=" * 100)
print("THREE-SEED SUMMARY")
print("=" * 100)

summary = (
    cost_threshold_df
    .groupby(["model", "r"])
    .agg(
        val_opt_t_mean=("val_opt_t", "mean"),
        val_opt_t_sd=("val_opt_t", "std"),

        theory_norm_cost_mean=(
            "theory_test_norm_cost",
            "mean"
        ),

        valopt_norm_cost_mean=(
            "valopt_test_norm_cost",
            "mean"
        ),

        theory_fnr_mean=(
            "theory_test_fnr",
            "mean"
        ),

        valopt_fnr_mean=(
            "valopt_test_fnr",
            "mean"
        ),

        theory_fpr_mean=(
            "theory_test_fpr",
            "mean"
        ),

        valopt_fpr_mean=(
            "valopt_test_fpr",
            "mean"
        ),

        theory_reject_mean=(
            "theory_test_reject",
            "mean"
        ),

        valopt_reject_mean=(
            "valopt_test_reject",
            "mean"
        )
    )
    .reset_index()
)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:

W4 COST-SENSITIVE THRESHOLDS — PER SEED
       model  seed  r  theoretical_t  val_opt_t  theory_test_norm_cost  valopt_test_norm_cost  theory_test_fnr  valopt_test_fnr  theory_test_fpr  valopt_test_fpr  theory_test_reject  valopt_test_reject
SGD Logistic    17  5       0.166667   0.166453               0.683313               0.683328         0.346720         0.346189         0.293181         0.293657            0.346605            0.347090
SGD Logistic    17 10       0.090909   0.105561               0.811723               0.823718         0.211509         0.235500         0.443263         0.413466            0.494481            0.465546
SGD Logistic    17 20       0.047619   0.059853               0.988732               1.044305         0.108715         0.145308         0.609956         0.538036            0.651694            0.585015
SGD Logistic    42  5       0.166667   0.157031               0.673246               0.671576         0.330398         0.308381         0.298630        

In [35]:
# ============================================================
# W4 - Step 31: Bootstrap uncertainty for MLP cost thresholds
#
# Includes:
#   - validation sampling variability
#   - validation-selected threshold variability
#   - independent test sampling variability
#   - all three MLP seeds
#
# Uses already validation-recalibrated probabilities.
# ============================================================

import numpy as np
import pandas as pd


B = 500
BOOT_SEED = 26031

rng = np.random.default_rng(
    BOOT_SEED
)

bootstrap_rows = []

n_val = len(y_val)
n_test = len(y_test)


for b in range(B):

    # --------------------------------------------------------
    # Bootstrap validation and test independently.
    # Same sampled loans are used across seeds so comparisons
    # remain paired.
    # --------------------------------------------------------
    val_idx = rng.integers(
        0,
        n_val,
        size=n_val
    )

    test_idx = rng.integers(
        0,
        n_test,
        size=n_test
    )

    yv = y_val[val_idx]
    yt = y_test[test_idx]

    for r in COST_RATIOS:

        theoretical_t = 1 / (1 + r)

        seed_thresholds = []
        seed_theory_costs = []
        seed_valopt_costs = []

        seed_theory_fnr = []
        seed_valopt_fnr = []

        seed_theory_fpr = []
        seed_valopt_fpr = []

        for seed in SEEDS:

            p_val = (
                cost_calibrated_predictions[
                    "MLP"
                ][seed]["val"][val_idx]
            )

            p_test = (
                cost_calibrated_predictions[
                    "MLP"
                ][seed]["test"][test_idx]
            )

            # -----------------------------------------------
            # Threshold is re-selected INSIDE each bootstrap
            # validation sample.
            # -----------------------------------------------
            val_opt_t = exact_optimal_threshold(
                yv,
                p_val,
                r
            )

            theory_res = evaluate_cost(
                yt,
                p_test,
                theoretical_t,
                r
            )

            valopt_res = evaluate_cost(
                yt,
                p_test,
                val_opt_t,
                r
            )

            seed_thresholds.append(
                val_opt_t
            )

            seed_theory_costs.append(
                theory_res[
                    "normalized_cost"
                ]
            )

            seed_valopt_costs.append(
                valopt_res[
                    "normalized_cost"
                ]
            )

            seed_theory_fnr.append(
                theory_res["fnr"]
            )

            seed_valopt_fnr.append(
                valopt_res["fnr"]
            )

            seed_theory_fpr.append(
                theory_res["fpr"]
            )

            seed_valopt_fpr.append(
                valopt_res["fpr"]
            )

        theory_cost_mean = np.mean(
            seed_theory_costs
        )

        valopt_cost_mean = np.mean(
            seed_valopt_costs
        )

        bootstrap_rows.append({
            "bootstrap": b + 1,
            "r": r,

            "val_opt_t_mean":
                np.mean(seed_thresholds),

            "val_opt_t_sd_across_seeds":
                np.std(
                    seed_thresholds,
                    ddof=1
                ),

            "theory_norm_cost":
                theory_cost_mean,

            "valopt_norm_cost":
                valopt_cost_mean,

            # Negative means validation-selected threshold
            # had lower test cost than theoretical threshold.
            "delta_cost_valopt_minus_theory":
                valopt_cost_mean
                - theory_cost_mean,

            "theory_fnr":
                np.mean(seed_theory_fnr),

            "valopt_fnr":
                np.mean(seed_valopt_fnr),

            "theory_fpr":
                np.mean(seed_theory_fpr),

            "valopt_fpr":
                np.mean(seed_valopt_fpr)
        })

    if (b + 1) % 50 == 0:
        print(
            f"Completed bootstrap "
            f"{b + 1}/{B}"
        )


mlp_cost_boot_df = pd.DataFrame(
    bootstrap_rows
)


# ------------------------------------------------------------
# Percentile summaries
# ------------------------------------------------------------

def bootstrap_summary(x):
    x = np.asarray(x)

    return pd.Series({
        "mean": np.mean(x),
        "median": np.median(x),
        "ci_low": np.quantile(
            x,
            0.025
        ),
        "ci_high": np.quantile(
            x,
            0.975
        )
    })


print("\n" + "=" * 100)
print("MLP BOOTSTRAP COST ANALYSIS")
print("500 bootstrap replicates")
print("=" * 100)


for r in COST_RATIOS:

    x = mlp_cost_boot_df[
        mlp_cost_boot_df["r"] == r
    ]

    print(
        f"\n{'=' * 30} "
        f"COST RATIO r={r} "
        f"{'=' * 30}"
    )

    metrics = [
        "val_opt_t_mean",
        "theory_norm_cost",
        "valopt_norm_cost",
        "delta_cost_valopt_minus_theory",
        "theory_fnr",
        "valopt_fnr",
        "theory_fpr",
        "valopt_fpr"
    ]

    for metric in metrics:

        s = bootstrap_summary(
            x[metric]
        )

        print(
            f"{metric:34s} "
            f"mean={s['mean']:.6f}  "
            f"95% CI=["
            f"{s['ci_low']:.6f}, "
            f"{s['ci_high']:.6f}]"
        )

    # Probability that the policy beats
    # the best trivial strategy.
    p_theory_better_trivial = np.mean(
        x["theory_norm_cost"] < 1
    )

    p_valopt_better_trivial = np.mean(
        x["valopt_norm_cost"] < 1
    )

    # Probability that validation-selected
    # threshold beats theoretical threshold.
    p_valopt_better_theory = np.mean(
        x[
            "delta_cost_valopt_minus_theory"
        ] < 0
    )

    print(
        "\nBootstrap proportion with "
        "normalized cost < 1:"
    )

    print(
        f"  Theoretical threshold : "
        f"{p_theory_better_trivial:.3f}"
    )

    print(
        f"  Validation-selected   : "
        f"{p_valopt_better_trivial:.3f}"
    )

    print(
        "Bootstrap proportion where "
        "validation-selected threshold "
        "beats theoretical threshold:"
    )

    print(
        f"  {p_valopt_better_theory:.3f}"
    )

Completed bootstrap 50/500
Completed bootstrap 100/500
Completed bootstrap 150/500
Completed bootstrap 200/500
Completed bootstrap 250/500
Completed bootstrap 300/500
Completed bootstrap 350/500
Completed bootstrap 400/500
Completed bootstrap 450/500
Completed bootstrap 500/500

MLP BOOTSTRAP COST ANALYSIS
500 bootstrap replicates

============================== COST RATIO r=5 ==============================
val_opt_t_mean                     mean=0.183010  95% CI=[0.172125, 0.196193]
theory_norm_cost                   mean=0.629436  95% CI=[0.619724, 0.639646]
valopt_norm_cost                   mean=0.631041  95% CI=[0.621118, 0.641073]
delta_cost_valopt_minus_theory     mean=0.001605  95% CI=[-0.000763, 0.004852]
theory_fnr                         mean=0.309609  95% CI=[0.301935, 0.317149]
valopt_fnr                         mean=0.346475  95% CI=[0.320175, 0.378664]
theory_fpr                         mean=0.278408  95% CI=[0.275378, 0.281127]
valopt_fpr                         mean=0.

In [36]:
# ============================================================
# W4 - Step 32: Decision-cost curve over a continuum of r
# Selected model family: MLP
#
# 500 bootstrap replicates
# Validation threshold is re-selected INSIDE every bootstrap
# replicate for every cost ratio.
#
# Normalized cost < 1  -> better than best trivial policy
# Cost reduction > 0  -> benefit relative to trivial policy
# ============================================================

import numpy as np
import pandas as pd


R_CURVE = np.array(
    [2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 20, 22, 25],
    dtype=float
)

THEORY_THRESHOLDS = 1 / (1 + R_CURVE)

B_CURVE = 500
CURVE_BOOT_SEED = 26032

rng_curve = np.random.default_rng(
    CURVE_BOOT_SEED
)


def optimal_thresholds_many_r(y, p, r_values):
    """
    Exact validation-optimal threshold for several cost ratios
    after sorting probabilities only once.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    r_values = np.asarray(r_values, dtype=float)

    order = np.argsort(-p)

    ps = p[order]
    ys = y[order]

    total_pos = ys.sum()

    cum_tp = np.cumsum(ys)
    cum_fp = np.cumsum(1 - ys)

    # End of each group of tied probabilities
    distinct_end = np.r_[
        ps[:-1] != ps[1:],
        True
    ]

    idx = np.where(
        distinct_end
    )[0]

    thresholds = ps[idx]

    tp = cum_tp[idx]
    fp = cum_fp[idx]
    fn = total_pos - tp

    # Rows = cost ratios
    # Columns = threshold candidates
    costs = (
        r_values[:, None]
        * fn[None, :]
        + fp[None, :]
    )

    best_candidate = np.argmin(
        costs,
        axis=1
    )

    best_thresholds = thresholds[
        best_candidate
    ].astype(float)

    best_costs = costs[
        np.arange(len(r_values)),
        best_candidate
    ]

    # Also permit "flag nobody"
    none_costs = (
        r_values * total_pos
    )

    use_none = (
        none_costs < best_costs
    )

    best_thresholds[
        use_none
    ] = 1.0

    return best_thresholds


def normalized_costs_many_thresholds(
    y,
    p,
    thresholds,
    r_values
):
    """
    Evaluate several thresholds efficiently after sorting
    test probabilities only once.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    thresholds = np.asarray(
        thresholds,
        dtype=float
    )

    r_values = np.asarray(
        r_values,
        dtype=float
    )

    order = np.argsort(p)

    ps = p[order]
    ys = y[order]

    n = len(y)

    prefix_pos = np.r_[
        0,
        np.cumsum(ys)
    ]

    total_pos = prefix_pos[-1]

    # First position with p >= threshold
    k = np.searchsorted(
        ps,
        thresholds,
        side="left"
    )

    tp = (
        total_pos
        - prefix_pos[k]
    )

    predicted_positive = (
        n - k
    )

    fp = (
        predicted_positive
        - tp
    )

    fn = (
        total_pos
        - tp
    )

    raw_cost = (
        r_values * fn + fp
    ) / n

    prevalence = (
        total_pos / n
    )

    approve_all_cost = (
        r_values * prevalence
    )

    reject_all_cost = (
        1 - prevalence
    )

    no_model_cost = np.minimum(
        approve_all_cost,
        reject_all_cost
    )

    normalized_cost = (
        raw_cost / no_model_cost
    )

    return normalized_cost


curve_rows = []

n_val = len(y_val)
n_test = len(y_test)


for b in range(B_CURVE):

    # Same bootstrap observations across all three seeds
    val_idx = rng_curve.integers(
        0,
        n_val,
        size=n_val
    )

    test_idx = rng_curve.integers(
        0,
        n_test,
        size=n_test
    )

    yv = y_val[val_idx]
    yt = y_test[test_idx]

    seed_opt_thresholds = []
    seed_theory_costs = []
    seed_valopt_costs = []

    for seed in SEEDS:

        p_val = (
            cost_calibrated_predictions[
                "MLP"
            ][seed]["val"][val_idx]
        )

        p_test = (
            cost_calibrated_predictions[
                "MLP"
            ][seed]["test"][test_idx]
        )

        # Re-select threshold for EVERY r
        # inside bootstrap validation sample
        opt_thresholds = (
            optimal_thresholds_many_r(
                yv,
                p_val,
                R_CURVE
            )
        )

        theory_costs = (
            normalized_costs_many_thresholds(
                yt,
                p_test,
                THEORY_THRESHOLDS,
                R_CURVE
            )
        )

        valopt_costs = (
            normalized_costs_many_thresholds(
                yt,
                p_test,
                opt_thresholds,
                R_CURVE
            )
        )

        seed_opt_thresholds.append(
            opt_thresholds
        )

        seed_theory_costs.append(
            theory_costs
        )

        seed_valopt_costs.append(
            valopt_costs
        )


    # Mean over matched model seeds
    mean_opt_t = np.mean(
        seed_opt_thresholds,
        axis=0
    )

    mean_theory_cost = np.mean(
        seed_theory_costs,
        axis=0
    )

    mean_valopt_cost = np.mean(
        seed_valopt_costs,
        axis=0
    )


    for j, r in enumerate(R_CURVE):

        curve_rows.append({
            "bootstrap": b + 1,
            "r": r,

            "theoretical_t":
                THEORY_THRESHOLDS[j],

            "val_opt_t":
                mean_opt_t[j],

            "theory_norm_cost":
                mean_theory_cost[j],

            "valopt_norm_cost":
                mean_valopt_cost[j],

            "theory_cost_reduction":
                1 - mean_theory_cost[j],

            "valopt_cost_reduction":
                1 - mean_valopt_cost[j],

            "delta_valopt_minus_theory":
                mean_valopt_cost[j]
                - mean_theory_cost[j]
        })

    if (b + 1) % 50 == 0:
        print(
            f"Completed continuum bootstrap "
            f"{b + 1}/{B_CURVE}"
        )


mlp_curve_boot_df = pd.DataFrame(
    curve_rows
)


# ------------------------------------------------------------
# Build manuscript-ready continuum summary
# ------------------------------------------------------------

summary_rows = []

for r in R_CURVE:

    x = mlp_curve_boot_df[
        mlp_curve_boot_df["r"] == r
    ]

    summary_rows.append({

        "r": r,

        "theoretical_t":
            1 / (1 + r),

        "val_opt_t_mean":
            x["val_opt_t"].mean(),

        "val_opt_t_ci_low":
            x["val_opt_t"].quantile(0.025),

        "val_opt_t_ci_high":
            x["val_opt_t"].quantile(0.975),

        "theory_norm_cost_mean":
            x["theory_norm_cost"].mean(),

        "theory_norm_cost_ci_low":
            x["theory_norm_cost"].quantile(0.025),

        "theory_norm_cost_ci_high":
            x["theory_norm_cost"].quantile(0.975),

        "valopt_norm_cost_mean":
            x["valopt_norm_cost"].mean(),

        "valopt_norm_cost_ci_low":
            x["valopt_norm_cost"].quantile(0.025),

        "valopt_norm_cost_ci_high":
            x["valopt_norm_cost"].quantile(0.975),

        "theory_cost_reduction_mean":
            x["theory_cost_reduction"].mean(),

        "valopt_cost_reduction_mean":
            x["valopt_cost_reduction"].mean(),

        "delta_mean":
            x[
                "delta_valopt_minus_theory"
            ].mean(),

        "delta_ci_low":
            x[
                "delta_valopt_minus_theory"
            ].quantile(0.025),

        "delta_ci_high":
            x[
                "delta_valopt_minus_theory"
            ].quantile(0.975),

        "p_theory_better_baseline":
            np.mean(
                x["theory_norm_cost"] < 1
            ),

        "p_valopt_better_baseline":
            np.mean(
                x["valopt_norm_cost"] < 1
            )
    })


mlp_curve_summary_df = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 130)
print("MLP DECISION-COST CONTINUUM WITH 95% BOOTSTRAP INTERVALS")
print("=" * 130)

print(
    mlp_curve_summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Completed continuum bootstrap 50/500
Completed continuum bootstrap 100/500
Completed continuum bootstrap 150/500
Completed continuum bootstrap 200/500
Completed continuum bootstrap 250/500
Completed continuum bootstrap 300/500
Completed continuum bootstrap 350/500
Completed continuum bootstrap 400/500
Completed continuum bootstrap 450/500
Completed continuum bootstrap 500/500

MLP DECISION-COST CONTINUUM WITH 95% BOOTSTRAP INTERVALS
        r  theoretical_t  val_opt_t_mean  val_opt_t_ci_low  val_opt_t_ci_high  theory_norm_cost_mean  theory_norm_cost_ci_low  theory_norm_cost_ci_high  valopt_norm_cost_mean  valopt_norm_cost_ci_low  valopt_norm_cost_ci_high  theory_cost_reduction_mean  valopt_cost_reduction_mean  delta_mean  delta_ci_low  delta_ci_high  p_theory_better_baseline  p_valopt_better_baseline
 2.000000       0.333333        0.327912          0.318192           0.338616               0.887562                 0.879676                  0.895795               0.886046              

In [37]:
# ============================================================
# W5 - Step 33: Predeclared grade-slice audit
#
# Selected family: validation-recalibrated MLP
# All observed LendingClub grade categories are reported.
# No post-hoc "worst slice" selection.
#
# Operating point:
#   r = 10
#   theoretical calibrated threshold = 1 / (1 + 10)
#
# Reports:
#   n
#   defaults
#   observed default prevalence + Wilson 95% CI
#   mean predicted risk
#   calibration gap
#   normalized decision cost
#   cost reduction vs best trivial policy
#   FNR
#   rejection rate
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import norm


# ------------------------------------------------------------
# Align grade labels with the existing test prediction arrays
# ------------------------------------------------------------

test_grade = (
    df_fixed36.loc[
        df_fixed36["split"] == "Test",
        "grade"
    ]
    .astype(str)
    .reset_index(drop=True)
)

assert len(test_grade) == len(y_test)


# ------------------------------------------------------------
# Wilson confidence interval for observed default prevalence
# ------------------------------------------------------------

def wilson_ci(k, n, alpha=0.05):

    if n == 0:
        return np.nan, np.nan

    z = norm.ppf(
        1 - alpha / 2
    )

    p = k / n

    denominator = (
        1 + z**2 / n
    )

    centre = (
        p + z**2 / (2 * n)
    ) / denominator

    half_width = (
        z
        * np.sqrt(
            p * (1 - p) / n
            + z**2 / (4 * n**2)
        )
        / denominator
    )

    return (
        centre - half_width,
        centre + half_width
    )


# ------------------------------------------------------------
# Predeclared slice definition
# ------------------------------------------------------------

grade_levels = sorted(
    test_grade.unique()
)

print(
    "NUMBER OF PREDECLARED GRADE SLICES:",
    len(grade_levels)
)

print(
    "GRADE SLICES:",
    grade_levels
)


# ------------------------------------------------------------
# r = 10 calibrated decision threshold
# ------------------------------------------------------------

R_SLICE = 10

T_SLICE = 1 / (
    1 + R_SLICE
)

print(
    "\nCost ratio r:",
    R_SLICE
)

print(
    "Theoretical threshold:",
    T_SLICE
)


# ------------------------------------------------------------
# Evaluate every grade, not merely the worst-performing one
# ------------------------------------------------------------

slice_rows = []


for grade in grade_levels:

    mask = (
        test_grade.to_numpy()
        == grade
    )

    y_slice = y_test[mask]

    n = len(y_slice)
    defaults = int(
        y_slice.sum()
    )

    observed_rate = (
        y_slice.mean()
    )

    ci_low, ci_high = wilson_ci(
        defaults,
        n
    )


    seed_mean_predictions = []
    seed_norm_costs = []
    seed_cost_reductions = []
    seed_fnrs = []
    seed_fprs = []
    seed_rejection_rates = []


    for seed in SEEDS:

        p_slice = (
            calibrated_predictions[
                "MLP"
            ][seed][mask]
        )

        seed_mean_predictions.append(
            p_slice.mean()
        )

        cost_result = evaluate_cost(
            y_slice,
            p_slice,
            T_SLICE,
            R_SLICE
        )

        norm_cost = (
            cost_result[
                "normalized_cost"
            ]
        )

        seed_norm_costs.append(
            norm_cost
        )

        seed_cost_reductions.append(
            1 - norm_cost
        )

        seed_fnrs.append(
            cost_result["fnr"]
        )

        seed_fprs.append(
            cost_result["fpr"]
        )

        seed_rejection_rates.append(
            cost_result[
                "rejection_rate"
            ]
        )


    mean_pred = np.mean(
        seed_mean_predictions
    )


    slice_rows.append({

        "grade":
            grade,

        "n":
            n,

        "defaults":
            defaults,

        "observed_rate":
            observed_rate,

        "observed_ci_low":
            ci_low,

        "observed_ci_high":
            ci_high,

        "mean_pred":
            mean_pred,

        # Positive = over-prediction
        # Negative = under-prediction
        "calibration_gap_pred_minus_obs":
            mean_pred
            - observed_rate,

        "norm_cost_mean":
            np.mean(
                seed_norm_costs
            ),

        "norm_cost_sd":
            np.std(
                seed_norm_costs,
                ddof=1
            ),

        # Positive = model beats best trivial
        # policy within that grade.
        "cost_reduction_mean":
            np.mean(
                seed_cost_reductions
            ),

        "fnr_mean":
            np.mean(
                seed_fnrs
            ),

        "fpr_mean":
            np.mean(
                seed_fprs
            ),

        "rejection_rate_mean":
            np.mean(
                seed_rejection_rates
            )
    })


grade_slice_df = pd.DataFrame(
    slice_rows
)


print(
    "\n" + "=" * 125
)

print(
    "PREDECLARED MLP GRADE-SLICE AUDIT "
    "AT r=10"
)

print(
    "=" * 125
)


print(
    grade_slice_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print(
    "\nINTERPRETATION GUIDE"
)

print(
    "normalized cost < 1  : "
    "model policy beats best trivial policy"
)

print(
    "cost reduction > 0   : "
    "positive decision value"
)

print(
    "calibration gap < 0  : "
    "mean predicted risk is below observed prevalence"
)

print(
    "calibration gap > 0  : "
    "mean predicted risk is above observed prevalence"
)

NUMBER OF PREDECLARED GRADE SLICES: 7
GRADE SLICES: ['A', 'B', 'C', 'D', 'E', 'F', 'G']

Cost ratio r: 10
Theoretical threshold: 0.09090909090909091

PREDECLARED MLP GRADE-SLICE AUDIT AT r=10
grade     n  defaults  observed_rate  observed_ci_low  observed_ci_high  mean_pred  calibration_gap_pred_minus_obs  norm_cost_mean  norm_cost_sd  cost_reduction_mean  fnr_mean  fpr_mean  rejection_rate_mean
    A 21784      1153       0.052929         0.050034          0.055982   0.057492                        0.004563        0.845042      0.007723             0.154958  0.427869  0.233144             0.251086
    B 32135      3899       0.121332         0.117807          0.124947   0.111916                       -0.009416        0.775854      0.003748             0.224146  0.239378  0.445306             0.483564
    C 23485      4730       0.201405         0.196325          0.206583   0.164298                       -0.037107        0.961930      0.008147             0.038070  0.182171  0.502497  

In [38]:
# ============================================================
# W5 - Step 34: Bootstrap uncertainty for all 7
# predeclared grade slices
#
# Selected model: validation-recalibrated MLP
# Operating point: theoretical r=10 threshold
#
# Bootstrap is performed WITHIN each grade.
# The same resampled loans are used across all 3 seeds.
#
# Produces 95% CIs for:
#   observed prevalence
#   mean predicted risk
#   calibration gap
#   normalized cost
#   cost reduction
#   FNR
#   rejection rate
# ============================================================

import numpy as np
import pandas as pd


B_SLICE = 1000
SLICE_BOOT_SEED = 26034

rng_slice = np.random.default_rng(
    SLICE_BOOT_SEED
)

grade_array = test_grade.to_numpy()

slice_boot_rows = []


for grade in grade_levels:

    grade_idx = np.where(
        grade_array == grade
    )[0]

    n_grade = len(
        grade_idx
    )

    print(
        f"\nBootstrapping Grade {grade} "
        f"(n={n_grade})"
    )

    for b in range(B_SLICE):

        # ----------------------------------------------------
        # Bootstrap WITHIN the grade slice
        # ----------------------------------------------------
        local_sample = rng_slice.integers(
            0,
            n_grade,
            size=n_grade
        )

        boot_idx = grade_idx[
            local_sample
        ]

        yb = y_test[
            boot_idx
        ]

        observed_rate = (
            yb.mean()
        )

        seed_mean_pred = []
        seed_norm_cost = []
        seed_cost_reduction = []
        seed_fnr = []
        seed_rejection = []

        for seed in SEEDS:

            pb = (
                calibrated_predictions[
                    "MLP"
                ][seed][boot_idx]
            )

            seed_mean_pred.append(
                pb.mean()
            )

            result = evaluate_cost(
                yb,
                pb,
                T_SLICE,
                R_SLICE
            )

            nc = result[
                "normalized_cost"
            ]

            seed_norm_cost.append(
                nc
            )

            seed_cost_reduction.append(
                1 - nc
            )

            seed_fnr.append(
                result["fnr"]
            )

            seed_rejection.append(
                result[
                    "rejection_rate"
                ]
            )


        mean_pred = np.mean(
            seed_mean_pred
        )

        slice_boot_rows.append({

            "grade":
                grade,

            "bootstrap":
                b + 1,

            "observed_rate":
                observed_rate,

            "mean_pred":
                mean_pred,

            "calibration_gap":
                mean_pred
                - observed_rate,

            "normalized_cost":
                np.mean(
                    seed_norm_cost
                ),

            "cost_reduction":
                np.mean(
                    seed_cost_reduction
                ),

            "fnr":
                np.nanmean(
                    seed_fnr
                ),

            "rejection_rate":
                np.mean(
                    seed_rejection
                )
        })

    print(
        f"Completed Grade {grade}"
    )


grade_slice_boot_df = pd.DataFrame(
    slice_boot_rows
)


# ------------------------------------------------------------
# Combine original point estimates with bootstrap CIs
# ------------------------------------------------------------

slice_ci_rows = []


for grade in grade_levels:

    boot = grade_slice_boot_df[
        grade_slice_boot_df[
            "grade"
        ] == grade
    ]

    point = grade_slice_df[
        grade_slice_df[
            "grade"
        ] == grade
    ].iloc[0]


    def ci(series):

        x = np.asarray(
            series,
            dtype=float
        )

        x = x[
            np.isfinite(x)
        ]

        return (
            np.quantile(
                x,
                0.025
            ),
            np.quantile(
                x,
                0.975
            )
        )


    obs_lo, obs_hi = ci(
        boot["observed_rate"]
    )

    pred_lo, pred_hi = ci(
        boot["mean_pred"]
    )

    gap_lo, gap_hi = ci(
        boot["calibration_gap"]
    )

    cost_lo, cost_hi = ci(
        boot["normalized_cost"]
    )

    reduction_lo, reduction_hi = ci(
        boot["cost_reduction"]
    )

    fnr_lo, fnr_hi = ci(
        boot["fnr"]
    )

    reject_lo, reject_hi = ci(
        boot["rejection_rate"]
    )


    slice_ci_rows.append({

        "grade":
            grade,

        "n":
            int(
                point["n"]
            ),

        "defaults":
            int(
                point["defaults"]
            ),

        "observed_rate":
            point[
                "observed_rate"
            ],

        "obs_ci_low":
            obs_lo,

        "obs_ci_high":
            obs_hi,

        "mean_pred":
            point[
                "mean_pred"
            ],

        "pred_ci_low":
            pred_lo,

        "pred_ci_high":
            pred_hi,

        "calibration_gap":
            point[
                "calibration_gap_pred_minus_obs"
            ],

        "gap_ci_low":
            gap_lo,

        "gap_ci_high":
            gap_hi,

        "normalized_cost":
            point[
                "norm_cost_mean"
            ],

        "cost_ci_low":
            cost_lo,

        "cost_ci_high":
            cost_hi,

        "cost_reduction":
            point[
                "cost_reduction_mean"
            ],

        "reduction_ci_low":
            reduction_lo,

        "reduction_ci_high":
            reduction_hi,

        "fnr":
            point[
                "fnr_mean"
            ],

        "fnr_ci_low":
            fnr_lo,

        "fnr_ci_high":
            fnr_hi,

        "rejection_rate":
            point[
                "rejection_rate_mean"
            ],

        "reject_ci_low":
            reject_lo,

        "reject_ci_high":
            reject_hi,

        # Bootstrap probability that model policy
        # beats best trivial policy in this grade.
        "p_cost_below_1":
            np.mean(
                boot[
                    "normalized_cost"
                ] < 1
            )
    })


grade_slice_ci_df = pd.DataFrame(
    slice_ci_rows
)


print(
    "\n" + "=" * 155
)

print(
    "PREDECLARED GRADE-SLICE AUDIT WITH "
    "95% BOOTSTRAP INTERVALS"
)

print(
    "=" * 155
)


print(
    grade_slice_ci_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


Bootstrapping Grade A (n=21784)
Completed Grade A

Bootstrapping Grade B (n=32135)
Completed Grade B

Bootstrapping Grade C (n=23485)
Completed Grade C

Bootstrapping Grade D (n=8618)
Completed Grade D

Bootstrapping Grade E (n=2275)
Completed Grade E

Bootstrapping Grade F (n=390)
Completed Grade F

Bootstrapping Grade G (n=97)
Completed Grade G

PREDECLARED GRADE-SLICE AUDIT WITH 95% BOOTSTRAP INTERVALS
grade     n  defaults  observed_rate  obs_ci_low  obs_ci_high  mean_pred  pred_ci_low  pred_ci_high  calibration_gap  gap_ci_low  gap_ci_high  normalized_cost  cost_ci_low  cost_ci_high  cost_reduction  reduction_ci_low  reduction_ci_high      fnr  fnr_ci_low  fnr_ci_high  rejection_rate  reject_ci_low  reject_ci_high  p_cost_below_1
    A 21784      1153       0.052929    0.049945     0.055913   0.057492     0.056652      0.058378         0.004563    0.001692     0.007357         0.845042     0.808564      0.882460        0.154958          0.117540           0.191436 0.427869    0.4

In [40]:
# ============================================================
# W6 - Step 35B: Read macro-construction Stata code
# directly from the original ZIP archive
# ============================================================

import zipfile
from pathlib import Path


ZIP_PATH = Path(
    r"E:\Governance-Policy\wb3ndt69gf-3.zip"
)

DO_MEMBERS = [
    "P2P_Macro_Codes.do",
    "Labels.do"
]

KEYWORDS = [
    "unemployment",
    "unem_rate",
    "merge",
    "join",
    "state",
    "addr_state",
    "issue_date",
    "issue_d",
    "daily_date",
    "month",
    "fred",
    "gdp",
    "cpi",
    "fedfunds",
    "riskprem",
    "lag",
    "l."
]


print("ZIP exists:", ZIP_PATH.exists())


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as z:

    members = z.namelist()

    print("\nZIP MEMBERS:")
    for m in members:
        print(m)

    for member in DO_MEMBERS:

        print("\n" + "=" * 100)
        print(member)
        print("=" * 100)

        if member not in members:
            print("NOT FOUND IN ZIP")
            continue

        raw = z.read(member)

        text = raw.decode(
            "utf-8",
            errors="ignore"
        )

        lines = text.splitlines()

        hit_lines = []

        for i, line in enumerate(lines):

            lower = line.lower()

            if any(
                keyword.lower() in lower
                for keyword in KEYWORDS
            ):
                hit_lines.append(i)

        if not hit_lines:
            print(
                "No macro-related keyword "
                "matches found."
            )
            continue

        printed = set()

        for i in hit_lines:

            start = max(
                0,
                i - 3
            )

            end = min(
                len(lines),
                i + 4
            )

            for j in range(
                start,
                end
            ):

                if j not in printed:

                    print(
                        f"{j + 1:4d}: "
                        f"{lines[j]}"
                    )

                    printed.add(j)

            print("-" * 100)

ZIP exists: True

ZIP MEMBERS:
Labels.do
P2P_Macro_Codes.do
P2P_Macro_Data.zip
P2P_Macro_Data.dta

P2P_Macro_Codes.do
   8:     drop `var'
   9:      }
  10: }
  11: gen issue_date=monthly(issue_d, "MY",2050)
  12: format %tmMonth_CCYY issue_date
  13: gen last_date=monthly(last_pymnt_d, "MY",2050)
  14: format %tmMonth_CCYY last_date
----------------------------------------------------------------------------------------------------
  15: replace last_date=issue_date if last_date==.
----------------------------------------------------------------------------------------------------
  16: gen daily_date=dofm(last_date)
----------------------------------------------------------------------------------------------------
  17: format daily_date %d
----------------------------------------------------------------------------------------------------
  18: gen quarter=quarter(daily_date)
----------------------------------------------------------------------------------------------------
  19:

In [41]:
# ============================================================
# W6 / LEAKAGE AUDIT - Step 36
#
# 1. Confirm quarter/month are derived from last_date
# 2. Determine whether macro variables are keyed to:
#       issue_date
#       last_date
#       issue_date + state
#       last_date + state
#
# This tells us whether the macro predictors are genuinely
# available at origination or contain post-origination timing.
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# A. Confirm quarter/month leakage in current modelling frame
# ------------------------------------------------------------

issue_month = (
    df_fixed36["issue_date"]
    .dt.month
)

issue_quarter = (
    df_fixed36["issue_date"]
    .dt.quarter
)

last_month = (
    df_fixed36["last_date"]
    .dt.month
)

last_quarter = (
    df_fixed36["last_date"]
    .dt.quarter
)


print("=" * 90)
print("A. QUARTER / MONTH TIMING AUDIT")
print("=" * 90)

print(
    "month matches ISSUE month:",
    np.mean(
        df_fixed36["month"].astype(float)
        == issue_month.astype(float)
    )
)

print(
    "month matches LAST-PAYMENT month:",
    np.mean(
        df_fixed36["month"].astype(float)
        == last_month.astype(float)
    )
)

print(
    "quarter matches ISSUE quarter:",
    np.mean(
        df_fixed36["quarter"].astype(float)
        == issue_quarter.astype(float)
    )
)

print(
    "quarter matches LAST-PAYMENT quarter:",
    np.mean(
        df_fixed36["quarter"].astype(float)
        == last_quarter.astype(float)
    )
)


# ------------------------------------------------------------
# B. Load minimal 2015 audit frame including state
# ------------------------------------------------------------

MACRO_COLS = [
    "unemployment",
    "unem_rate",
    "gdpindex",
    "gdpreal",
    "gdpcontrib",
    "cpi",
    "inf",
    "FEDFUNDS",
    "gdppercap",
    "riskprem"
]

AUDIT_COLS = [
    "issue_date",
    "last_date",
    "term",
    "addr_state"
] + MACRO_COLS


macro_parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=AUDIT_COLS,
    convert_categoricals=False,
    chunksize=250_000
)


for i, chunk in enumerate(reader, start=1):

    term_months = (
        chunk["term"]
        .astype(str)
        .str.extract(
            r"(\d+)",
            expand=False
        )
        .astype(float)
    )

    x = chunk.loc[
        (term_months == 36)
        &
        (chunk["issue_date"].dt.year == 2015)
    ].copy()

    if len(x) > 0:
        macro_parts.append(x)

    print(
        f"Finished chunk {i}"
    )


macro_audit = pd.concat(
    macro_parts,
    ignore_index=True
)


print(
    "\n2015 36-month audit rows:",
    len(macro_audit)
)


# ------------------------------------------------------------
# C. Measure how deterministic each macro variable is
# under four possible timing/geographic keys.
#
# fraction_singleton = fraction of groups in which the macro
# variable has exactly one unique value.
#
# A value close to 1 indicates a plausible merge key.
# ------------------------------------------------------------

def singleton_fraction(
    df,
    group_cols,
    value_col
):

    x = (
        df
        .dropna(
            subset=group_cols
            + [value_col]
        )
        .groupby(
            group_cols,
            observed=True
        )[value_col]
        .nunique()
    )

    if len(x) == 0:
        return np.nan

    return np.mean(
        x == 1
    )


timing_rows = []


for c in MACRO_COLS:

    timing_rows.append({

        "variable":
            c,

        "issue_month_only":
            singleton_fraction(
                macro_audit,
                ["issue_date"],
                c
            ),

        "last_month_only":
            singleton_fraction(
                macro_audit,
                ["last_date"],
                c
            ),

        "issue_month_state":
            singleton_fraction(
                macro_audit,
                [
                    "issue_date",
                    "addr_state"
                ],
                c
            ),

        "last_month_state":
            singleton_fraction(
                macro_audit,
                [
                    "last_date",
                    "addr_state"
                ],
                c
            ),

        "overall_unique":
            macro_audit[c]
            .nunique(
                dropna=True
            )
    })


macro_timing_df = pd.DataFrame(
    timing_rows
)


print(
    "\n" + "=" * 90
)

print(
    "B. MACRO VARIABLE TIMING / GEOGRAPHY AUDIT"
)

print(
    "=" * 90
)


print(
    macro_timing_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

A. QUARTER / MONTH TIMING AUDIT
month matches ISSUE month: 0.257255697121932
month matches LAST-PAYMENT month: 1.0
quarter matches ISSUE quarter: 0.4372786974548316
quarter matches LAST-PAYMENT quarter: 1.0
Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11

2015 36-month audit rows: 283108

B. MACRO VARIABLE TIMING / GEOGRAPHY AUDIT
    variable  issue_month_only  last_month_only  issue_month_state  last_month_state  overall_unique
unemployment          0.000000         0.000000           0.000000          1.000000            1967
   unem_rate          0.000000         0.000000           0.000000          1.000000              55
    gdpindex          0.000000         0.000000           0.000000          1.000000             617
     gdpreal          0.000000         0.000000           0.000000          1.000000             622
  gdpcontrib         

In [42]:
# ============================================================
# W6 / LEAKAGE AUDIT - Step 37
#
# Reconstruct the macroeconomic state-month panel and test
# whether it can be safely re-aligned from:
#
#       last_date + state
#             TO
#       issue_date + state
#
# No model is retrained yet.
# ============================================================

import pandas as pd
import numpy as np


MACRO_COLS = [
    "unemployment",
    "unem_rate",
    "gdpindex",
    "gdpreal",
    "gdpcontrib",
    "cpi",
    "inf",
    "FEDFUNDS",
    "gdppercap",
    "riskprem"
]


# ------------------------------------------------------------
# A. Build candidate state-month macro lookup from FULL dataset
# ------------------------------------------------------------

lookup_parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "last_date",
        "addr_state"
    ] + MACRO_COLS,
    convert_categoricals=False,
    chunksize=250_000
)


for i, chunk in enumerate(reader, start=1):

    x = chunk.dropna(
        subset=[
            "last_date",
            "addr_state"
        ]
    ).copy()

    # Collapse repeated loans sharing identical
    # state-month macro values within each chunk.
    x = x.drop_duplicates(
        subset=[
            "last_date",
            "addr_state"
        ] + MACRO_COLS
    )

    lookup_parts.append(x)

    print(
        f"Lookup chunk {i} complete"
    )


macro_lookup_candidates = pd.concat(
    lookup_parts,
    ignore_index=True
)


print(
    "\nCandidate lookup rows:",
    len(macro_lookup_candidates)
)


# ------------------------------------------------------------
# B. Check whether each state-month has exactly ONE value
#    for each macro variable across the full dataset
# ------------------------------------------------------------

key_cols = [
    "last_date",
    "addr_state"
]


uniqueness = (
    macro_lookup_candidates
    .groupby(
        key_cols,
        observed=True
    )[MACRO_COLS]
    .nunique(
        dropna=False
    )
)


conflict_rows = []

for c in MACRO_COLS:

    n_conflict = int(
        (uniqueness[c] > 1).sum()
    )

    max_unique = int(
        uniqueness[c].max()
    )

    conflict_rows.append({
        "variable": c,
        "state_month_keys_with_conflict":
            n_conflict,
        "max_unique_values_per_key":
            max_unique
    })


macro_conflict_df = pd.DataFrame(
    conflict_rows
)


print(
    "\n" + "=" * 90
)

print(
    "FULL-DATA STATE-MONTH UNIQUENESS CHECK"
)

print(
    "=" * 90
)

print(
    macro_conflict_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# C. Create one state-month macro lookup
#    ONLY if values are unique within state-month
# ------------------------------------------------------------

macro_lookup = (
    macro_lookup_candidates
    .sort_values(
        key_cols
    )
    .drop_duplicates(
        subset=key_cols,
        keep="first"
    )
    .copy()
)


# Rename last_date -> issue_date because we now use the
# historical state-month value corresponding to origination.
macro_lookup = macro_lookup.rename(
    columns={
        "last_date":
            "issue_date",

        **{
            c: f"{c}_issue"
            for c in MACRO_COLS
        }
    }
)


print(
    "\nUnique state-month lookup rows:",
    len(macro_lookup)
)


# ------------------------------------------------------------
# D. Reload only state + issue date for our FINAL clean cohort
#    using the exact same chronological split.
# ------------------------------------------------------------

key_parts = []

reader = pd.read_stata(
    DTA_PATH,
    columns=[
        "issue_date",
        "term",
        "addr_state"
    ],
    convert_categoricals=False,
    chunksize=250_000
)


for i, chunk in enumerate(reader, start=1):

    term_months = (
        chunk["term"]
        .astype(str)
        .str.extract(
            r"(\d+)",
            expand=False
        )
        .astype(float)
    )

    x = chunk.loc[
        (term_months == 36)
        &
        (chunk["issue_date"].dt.year == 2015)
    ].copy()

    x["split"] = pd.NA

    x.loc[
        (x["issue_date"] >= "2015-01-01")
        &
        (x["issue_date"] < "2015-07-01"),
        "split"
    ] = "Train"

    x.loc[
        (x["issue_date"] >= "2015-07-01")
        &
        (x["issue_date"] < "2015-09-01"),
        "split"
    ] = "Validation"

    x.loc[
        (x["issue_date"] >= "2015-10-01")
        &
        (x["issue_date"] < "2016-01-01"),
        "split"
    ] = "Test"

    x = x.loc[
        x["split"].notna(),
        [
            "issue_date",
            "addr_state",
            "split"
        ]
    ]

    key_parts.append(x)

    print(
        f"Cohort-key chunk {i} complete"
    )


fixed36_keys = pd.concat(
    key_parts,
    ignore_index=True
)


print(
    "\nCohort key rows:",
    len(fixed36_keys)
)

print(
    "Expected model rows:",
    len(df_fixed36)
)


# Ensure exact row alignment with existing modelling frame
assert (
    len(fixed36_keys)
    == len(df_fixed36)
)

assert np.array_equal(
    fixed36_keys[
        "issue_date"
    ].to_numpy(),
    df_fixed36[
        "issue_date"
    ].to_numpy()
)

assert np.array_equal(
    fixed36_keys[
        "split"
    ].astype(str).to_numpy(),
    df_fixed36[
        "split"
    ].astype(str).to_numpy()
)


# ------------------------------------------------------------
# E. Join CORRECT origination-state-month macros
# ------------------------------------------------------------

fixed36_issue_macro = (
    fixed36_keys
    .reset_index()
    .merge(
        macro_lookup,
        on=[
            "issue_date",
            "addr_state"
        ],
        how="left",
        validate="many_to_one",
        sort=False
    )
    .sort_values(
        "index"
    )
    .reset_index(
        drop=True
    )
)


issue_macro_cols = [
    f"{c}_issue"
    for c in MACRO_COLS
]


coverage_rows = []

for c in issue_macro_cols:

    coverage_rows.append({
        "variable":
            c,

        "available":
            int(
                fixed36_issue_macro[
                    c
                ].notna().sum()
            ),

        "missing":
            int(
                fixed36_issue_macro[
                    c
                ].isna().sum()
            ),

        "coverage":
            fixed36_issue_macro[
                c
            ].notna().mean()
    })


macro_coverage_df = pd.DataFrame(
    coverage_rows
)


print(
    "\n" + "=" * 90
)

print(
    "ORIGINATION-TIME MACRO COVERAGE "
    "FOR CLEAN 2015 COHORT"
)

print(
    "=" * 90
)

print(
    macro_coverage_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


complete_all = (
    fixed36_issue_macro[
        issue_macro_cols
    ]
    .notna()
    .all(
        axis=1
    )
)


print(
    "\nRows with ALL 10 issue-time "
    "macro variables available:"
)

print(
    f"{complete_all.sum()} / "
    f"{len(complete_all)} "
    f"({complete_all.mean():.6%})"
)


print(
    "\nComplete macro coverage by split:"
)

coverage_by_split = (
    fixed36_issue_macro
    .assign(
        complete_macro=complete_all
    )
    .groupby(
        "split"
    )["complete_macro"]
    .agg([
        "count",
        "sum",
        "mean"
    ])
)

print(
    coverage_by_split
)

Lookup chunk 1 complete
Lookup chunk 2 complete
Lookup chunk 3 complete
Lookup chunk 4 complete
Lookup chunk 5 complete
Lookup chunk 6 complete
Lookup chunk 7 complete
Lookup chunk 8 complete
Lookup chunk 9 complete
Lookup chunk 10 complete
Lookup chunk 11 complete

Candidate lookup rows: 4684

FULL-DATA STATE-MONTH UNIQUENESS CHECK
    variable  state_month_keys_with_conflict  max_unique_values_per_key
unemployment                               0                          1
   unem_rate                               0                          1
    gdpindex                               0                          1
     gdpreal                               0                          1
  gdpcontrib                               0                          1
         cpi                               0                          1
         inf                               0                          1
    FEDFUNDS                               0                          1
   gdppercap     

In [43]:
# ============================================================
# W6 - Step 38: Characterise corrected unemployment variable
# and derive the TRUE training-period stress threshold
#
# Uses unemployment aligned to:
#       issue_date + addr_state
#
# Determines:
#   - national vs state-level structure
#   - within-month variation across states
#   - training-period 75th percentile
#   - test stress count / prevalence / default rate
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# A. How many distinct values exist across states
#    within the SAME origination month?
# ------------------------------------------------------------

macro_month_variation_rows = []

for c in issue_macro_cols:

    by_month = (
        fixed36_issue_macro
        .groupby(
            "issue_date",
            observed=True
        )[c]
        .nunique(
            dropna=True
        )
    )

    macro_month_variation_rows.append({

        "variable":
            c,

        "min_unique_per_month":
            int(
                by_month.min()
            ),

        "median_unique_per_month":
            float(
                by_month.median()
            ),

        "max_unique_per_month":
            int(
                by_month.max()
            ),

        # If this equals 1, variable is effectively
        # national/common across states that month.
        "months_with_one_value":
            int(
                (by_month == 1).sum()
            ),

        "months_total":
            int(
                len(by_month)
            )
    })


macro_month_variation_df = pd.DataFrame(
    macro_month_variation_rows
)


print("=" * 100)
print("A. WITHIN-MONTH VARIATION ACROSS STATES")
print("=" * 100)

print(
    macro_month_variation_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# B. Detailed unemployment-rate profile by issue month
# ------------------------------------------------------------

unem_month_profile = (
    fixed36_issue_macro
    .groupby(
        "issue_date",
        observed=True
    )["unem_rate_issue"]
    .agg(
        n_loans="size",
        unique_rates="nunique",
        min_rate="min",
        median_rate="median",
        max_rate="max"
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("B. CORRECTED UNEMPLOYMENT RATE BY ORIGINATION MONTH")
print("=" * 100)

print(
    unem_month_profile.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# C. Training-derived 75th percentile stress threshold
# ------------------------------------------------------------

train_macro_mask = (
    fixed36_issue_macro["split"]
    == "Train"
)

val_macro_mask = (
    fixed36_issue_macro["split"]
    == "Validation"
)

test_macro_mask = (
    fixed36_issue_macro["split"]
    == "Test"
)


train_unem = (
    fixed36_issue_macro.loc[
        train_macro_mask,
        "unem_rate_issue"
    ]
    .dropna()
)


stress_threshold = float(
    train_unem.quantile(
        0.75
    )
)


print("\n" + "=" * 100)
print("C. TRAINING-DERIVED STRESS THRESHOLD")
print("=" * 100)

print(
    f"Training unemployment observations: "
    f"{len(train_unem)}"
)

print(
    f"Training unem_rate min: "
    f"{train_unem.min():.6f}"
)

print(
    f"Training unem_rate median: "
    f"{train_unem.median():.6f}"
)

print(
    f"Training unem_rate 75th percentile: "
    f"{stress_threshold:.6f}"
)

print(
    f"Training unem_rate max: "
    f"{train_unem.max():.6f}"
)

print(
    f"\n75th percentile as percentage: "
    f"{100 * stress_threshold:.2f}%"
)


# ------------------------------------------------------------
# D. Apply frozen training threshold to validation/test
# ------------------------------------------------------------

stress_audit = fixed36_issue_macro[
    [
        "split",
        "issue_date",
        "addr_state",
        "unem_rate_issue"
    ]
].copy()


# Add outcome in exact row order
stress_audit["default"] = (
    df_fixed36[
        "default"
    ]
    .to_numpy()
)


stress_audit["stress"] = (
    stress_audit[
        "unem_rate_issue"
    ]
    >= stress_threshold
)


stress_summary = (
    stress_audit
    .groupby(
        ["split", "stress"],
        observed=True
    )
    .agg(
        n=("default", "size"),
        defaults=("default", "sum"),
        default_rate=("default", "mean"),
        mean_unem_rate=(
            "unem_rate_issue",
            "mean"
        ),
        min_unem_rate=(
            "unem_rate_issue",
            "min"
        ),
        max_unem_rate=(
            "unem_rate_issue",
            "max"
        )
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("D. STRESS COUNTS USING CORRECTED ISSUE-TIME UNEMPLOYMENT")
print("=" * 100)

print(
    stress_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# E. Test stress counts by origination month
# ------------------------------------------------------------

test_stress_month = (
    stress_audit.loc[
        stress_audit["split"]
        == "Test"
    ]
    .groupby(
        [
            "issue_date",
            "stress"
        ],
        observed=True
    )
    .agg(
        n=("default", "size"),
        defaults=("default", "sum"),
        default_rate=("default", "mean")
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("E. TEST STRESS DISTRIBUTION BY ORIGINATION MONTH")
print("=" * 100)

print(
    test_stress_month.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

A. WITHIN-MONTH VARIATION ACROSS STATES
          variable  min_unique_per_month  median_unique_per_month  max_unique_per_month  months_with_one_value  months_total
unemployment_issue                    46                     46.0                    49                      0            11
   unem_rate_issue                    25                     27.0                    32                      0            11
    gdpindex_issue                    46                     46.0                    49                      0            11
     gdpreal_issue                    46                     46.0                    49                      0            11
  gdpcontrib_issue                    33                     43.0                    44                      0            11
         cpi_issue                    15                     17.0                    18                      0            11
         inf_issue                    18                     20.0                    

In [44]:
# ============================================================
# LEAKAGE-FREE REBUILD - Step 39
#
# Correct:
#   1. macro variables -> issue_date + state values
#   2. month / quarter -> derived from issue_date
#
# Keep the same conceptual predictor set.
# Drop only:
#   term       (constant: all 36 months)
#   issue_year (constant: all 2015)
#
# No modelling yet.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# A. Start from the previously extracted clean cohort
# ------------------------------------------------------------

df_corrected = df_fixed36.copy()


# ------------------------------------------------------------
# B. Replace the 10 leaked last-date macro variables
#    with correctly aligned ISSUE-DATE + STATE values
# ------------------------------------------------------------

for c in MACRO_COLS:

    corrected_col = f"{c}_issue"

    df_corrected[c] = (
        fixed36_issue_macro[
            corrected_col
        ]
        .to_numpy()
    )


# ------------------------------------------------------------
# C. Replace leaked month / quarter with origination timing
# ------------------------------------------------------------

df_corrected["month"] = (
    df_corrected[
        "issue_date"
    ]
    .dt.month
    .astype(float)
)

df_corrected["quarter"] = (
    df_corrected[
        "issue_date"
    ]
    .dt.quarter
    .astype(float)
)


# ------------------------------------------------------------
# D. Final modelling feature list
# ------------------------------------------------------------

corrected_model_features = [
    c for c in planned_features
    if c not in [
        "term",
        "issue_year"
    ]
]


# ------------------------------------------------------------
# E. Build split matrices
# ------------------------------------------------------------

X_train_corrected = (
    df_corrected.loc[
        df_corrected["split"] == "Train",
        corrected_model_features
    ]
    .copy()
)

X_val_corrected = (
    df_corrected.loc[
        df_corrected["split"] == "Validation",
        corrected_model_features
    ]
    .copy()
)

X_test_corrected = (
    df_corrected.loc[
        df_corrected["split"] == "Test",
        corrected_model_features
    ]
    .copy()
)


# ------------------------------------------------------------
# F. Sanity checks
# ------------------------------------------------------------

print("=" * 95)
print("LEAKAGE-FREE FEATURE FRAME")
print("=" * 95)

print(
    "Final raw predictors:",
    len(corrected_model_features)
)

print(
    "\nTrain shape:",
    X_train_corrected.shape
)

print(
    "Validation shape:",
    X_val_corrected.shape
)

print(
    "Test shape:",
    X_test_corrected.shape
)


print(
    "\nCorrected month matches ISSUE month:"
)

print(
    np.mean(
        df_corrected["month"]
        ==
        df_corrected[
            "issue_date"
        ].dt.month
    )
)


print(
    "\nCorrected quarter matches ISSUE quarter:"
)

print(
    np.mean(
        df_corrected["quarter"]
        ==
        df_corrected[
            "issue_date"
        ].dt.quarter
    )
)


print(
    "\nMissing values by split "
    "across final predictors:"
)

for split_name, X in [
    ("Train", X_train_corrected),
    ("Validation", X_val_corrected),
    ("Test", X_test_corrected)
]:

    print(
        f"{split_name:10s}: "
        f"{int(X.isna().sum().sum())}"
    )


print(
    "\nTop missing predictors "
    "in corrected validation set:"
)

print(
    X_val_corrected
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(15)
)


print(
    "\nCorrected macro ranges "
    "by split:"
)

for split_name, X in [
    ("Train", X_train_corrected),
    ("Validation", X_val_corrected),
    ("Test", X_test_corrected)
]:

    print(
        f"\n{split_name}"
    )

    print(
        X[
            [
                "unem_rate",
                "FEDFUNDS",
                "riskprem"
            ]
        ]
        .agg([
            "min",
            "median",
            "max"
        ])
    )

LEAKAGE-FREE FEATURE FRAME
Final raw predictors: 39

Train shape: (120766, 39)
Validation shape: (54794, 39)
Test shape: (88784, 39)

Corrected month matches ISSUE month:
1.0

Corrected quarter matches ISSUE quarter:
1.0

Missing values by split across final predictors:
Train     : 47
Validation: 240
Test      : 38

Top missing predictors in corrected validation set:
revol_util        30
unem_rate         21
riskprem          21
FEDFUNDS          21
inf               21
cpi               21
unemployment      21
gdpcontrib        21
gdpreal           21
gdpindex          21
gdppercap         21
emp_length         0
dti                0
annual_inc         0
home_ownership     0
dtype: int64

Corrected macro ranges by split:

Train
        unem_rate  FEDFUNDS  riskprem
min         0.032      0.11    3.2075
median      0.057      0.12    3.2075
max         0.074      0.13    3.2075

Validation
        unem_rate  FEDFUNDS  riskprem
min         0.030      0.13    3.2075
median      0.053    

In [45]:
# ============================================================
# LEAKAGE-FREE REBUILD - Step 40
# Training-only preprocessing for corrected feature frame
#
# Automatically:
#   - detects/removes zero-variance training predictors
#   - identifies numeric/categorical columns
#   - median-imputes numeric variables using TRAIN only
#   - mode-imputes categorical variables using TRAIN only
#   - standardizes numeric variables
#   - one-hot encodes categorical variables
#   - safely handles unseen validation/test categories
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# A. Detect zero-variance predictors using TRAIN only
# ------------------------------------------------------------

train_nunique_corrected = (
    X_train_corrected
    .nunique(dropna=False)
)

constant_cols_corrected = (
    train_nunique_corrected[
        train_nunique_corrected <= 1
    ]
    .index
    .tolist()
)


print("=" * 90)
print("ZERO-VARIANCE TRAINING PREDICTORS")
print("=" * 90)

for c in constant_cols_corrected:

    values = (
        X_train_corrected[c]
        .drop_duplicates()
        .tolist()
    )

    print(
        f"{c:25s} "
        f"unique={train_nunique_corrected[c]} "
        f"value={values}"
    )


# ------------------------------------------------------------
# B. Final leakage-free predictor list
# ------------------------------------------------------------

final_model_features = [
    c for c in corrected_model_features
    if c not in constant_cols_corrected
]


X_train_final = (
    X_train_corrected[
        final_model_features
    ]
    .copy()
)

X_val_final = (
    X_val_corrected[
        final_model_features
    ]
    .copy()
)

X_test_final = (
    X_test_corrected[
        final_model_features
    ]
    .copy()
)


# ------------------------------------------------------------
# C. Detect numeric / categorical columns from TRAIN only
# ------------------------------------------------------------

final_cat_cols = (
    X_train_final
    .select_dtypes(
        include=[
            "object",
            "category"
        ]
    )
    .columns
    .tolist()
)

final_num_cols = [
    c for c in final_model_features
    if c not in final_cat_cols
]


print("\n" + "=" * 90)
print("FINAL LEAKAGE-FREE FEATURE SET")
print("=" * 90)

print(
    "Raw predictors:",
    len(final_model_features)
)

print(
    "Numeric predictors:",
    len(final_num_cols)
)

print(
    "Categorical predictors:",
    len(final_cat_cols)
)

print(
    "\nCategorical predictors:"
)

print(
    final_cat_cols
)


# ------------------------------------------------------------
# D. Check unseen categorical levels
# ------------------------------------------------------------

print(
    "\nUNSEEN CATEGORICAL LEVELS"
)

for c in final_cat_cols:

    train_levels = set(
        X_train_final[c]
        .dropna()
        .astype(str)
        .unique()
    )

    val_levels = set(
        X_val_final[c]
        .dropna()
        .astype(str)
        .unique()
    )

    test_levels = set(
        X_test_final[c]
        .dropna()
        .astype(str)
        .unique()
    )

    print(f"\n{c}")

    print(
        "  unseen val :",
        sorted(
            val_levels
            - train_levels
        )
    )

    print(
        "  unseen test:",
        sorted(
            test_levels
            - train_levels
        )
    )


# ------------------------------------------------------------
# E. Build TRAIN-fitted preprocessing pipeline
# ------------------------------------------------------------

numeric_pipe_corrected = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])


categorical_pipe_corrected = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


preprocessor_corrected = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipe_corrected,
            final_num_cols
        ),
        (
            "categorical",
            categorical_pipe_corrected,
            final_cat_cols
        )
    ],
    remainder="drop"
)


# ------------------------------------------------------------
# F. Fit ONLY on corrected training data
# ------------------------------------------------------------

X_train_proc_corrected = (
    preprocessor_corrected
    .fit_transform(
        X_train_final
    )
)

X_val_proc_corrected = (
    preprocessor_corrected
    .transform(
        X_val_final
    )
)

X_test_proc_corrected = (
    preprocessor_corrected
    .transform(
        X_test_final
    )
)


corrected_feature_names = (
    preprocessor_corrected
    .get_feature_names_out()
)


# ------------------------------------------------------------
# G. Final integrity checks
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TRANSFORMED LEAKAGE-FREE MATRICES")
print("=" * 90)

print(
    "Train:",
    X_train_proc_corrected.shape
)

print(
    "Validation:",
    X_val_proc_corrected.shape
)

print(
    "Test:",
    X_test_proc_corrected.shape
)

print(
    "\nEncoded feature count:",
    len(corrected_feature_names)
)

print(
    "\nAny NaN?"
)

print(
    "Train:",
    np.isnan(
        X_train_proc_corrected
    ).any()
)

print(
    "Validation:",
    np.isnan(
        X_val_proc_corrected
    ).any()
)

print(
    "Test:",
    np.isnan(
        X_test_proc_corrected
    ).any()
)

print(
    "\nAny infinite values?"
)

print(
    "Train:",
    np.isinf(
        X_train_proc_corrected
    ).any()
)

print(
    "Validation:",
    np.isinf(
        X_val_proc_corrected
    ).any()
)

print(
    "Test:",
    np.isinf(
        X_test_proc_corrected
    ).any()
)

print(
    "\nFirst 20 encoded features:"
)

for f in corrected_feature_names[:20]:
    print(f)

ZERO-VARIANCE TRAINING PREDICTORS
riskprem                  unique=1 value=[3.2075]

FINAL LEAKAGE-FREE FEATURE SET
Raw predictors: 38
Numeric predictors: 31
Categorical predictors: 7

Categorical predictors:
['emp_length', 'home_ownership', 'verification_status', 'grade', 'sub_grade', 'purpose', 'initial_list_status']

UNSEEN CATEGORICAL LEVELS

emp_length
  unseen val : []
  unseen test: []

home_ownership
  unseen val : []
  unseen test: ['ANY']

verification_status
  unseen val : []
  unseen test: []

grade
  unseen val : []
  unseen test: []

sub_grade
  unseen val : []
  unseen test: []

purpose
  unseen val : []
  unseen test: []

initial_list_status
  unseen val : []
  unseen test: []

TRANSFORMED LEAKAGE-FREE MATRICES
Train: (120766, 106)
Validation: (54794, 106)
Test: (88784, 106)

Encoded feature count: 106

Any NaN?
Train: False
Validation: False
Test: False

Any infinite values?
Train: False
Validation: False
Test: False

First 20 encoded features:
numeric__annual_inc
nume

In [46]:
# ============================================================
# LEAKAGE-FREE RERUN - Step 41
# Corrected SGD Logistic across 3 seeds
#
# Uses:
#   X_train_proc_corrected
#   X_val_proc_corrected
#   X_test_proc_corrected
#
# These results supersede the provisional SGD results.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)


SEEDS = [17, 42, 2026]

sgd_corrected_results = []
sgd_corrected_predictions = {}


for seed in SEEDS:

    print("\n" + "=" * 75)
    print(
        f"LEAKAGE-FREE SGD LOGISTIC - SEED {seed}"
    )
    print("=" * 75)

    model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-5,
        max_iter=2000,
        tol=1e-4,
        random_state=seed
    )

    model.fit(
        X_train_proc_corrected,
        y_train
    )


    p_val = model.predict_proba(
        X_val_proc_corrected
    )[:, 1]

    p_test = model.predict_proba(
        X_test_proc_corrected
    )[:, 1]


    # Numerical protection for log loss
    p_val = np.clip(
        p_val,
        1e-7,
        1 - 1e-7
    )

    p_test = np.clip(
        p_test,
        1e-7,
        1 - 1e-7
    )


    sgd_corrected_predictions[
        seed
    ] = {
        "val": p_val.copy(),
        "test": p_test.copy()
    }


    row = {

        "seed":
            seed,

        "val_auc":
            roc_auc_score(
                y_val,
                p_val
            ),

        "test_auc":
            roc_auc_score(
                y_test,
                p_test
            ),

        "val_ap":
            average_precision_score(
                y_val,
                p_val
            ),

        "test_ap":
            average_precision_score(
                y_test,
                p_test
            ),

        "val_brier":
            brier_score_loss(
                y_val,
                p_val
            ),

        "test_brier":
            brier_score_loss(
                y_test,
                p_test
            ),

        "val_logloss":
            log_loss(
                y_val,
                p_val
            ),

        "test_logloss":
            log_loss(
                y_test,
                p_test
            ),

        "n_iter":
            int(
                model.n_iter_
            )
    }


    sgd_corrected_results.append(
        row
    )


    print(
        "Iterations   :",
        model.n_iter_
    )

    print(
        f"Val AUC      : "
        f"{row['val_auc']:.6f}"
    )

    print(
        f"Test AUC     : "
        f"{row['test_auc']:.6f}"
    )

    print(
        f"Val AP       : "
        f"{row['val_ap']:.6f}"
    )

    print(
        f"Test AP      : "
        f"{row['test_ap']:.6f}"
    )

    print(
        f"Val Brier    : "
        f"{row['val_brier']:.6f}"
    )

    print(
        f"Test Brier   : "
        f"{row['test_brier']:.6f}"
    )

    print(
        f"Val LogLoss  : "
        f"{row['val_logloss']:.6f}"
    )

    print(
        f"Test LogLoss : "
        f"{row['test_logloss']:.6f}"
    )


sgd_corrected_results_df = pd.DataFrame(
    sgd_corrected_results
)


print("\n" + "=" * 75)
print(
    "LEAKAGE-FREE SGD LOGISTIC - THREE-SEED SUMMARY"
)
print("=" * 75)

print(
    sgd_corrected_results_df.to_string(
        index=False
    )
)


print("\nMEAN ± SD")

for metric in [
    "val_auc",
    "test_auc",
    "val_ap",
    "test_ap",
    "val_brier",
    "test_brier",
    "val_logloss",
    "test_logloss"
]:

    print(
        f"{metric:15s}: "
        f"{sgd_corrected_results_df[metric].mean():.6f} "
        f"± "
        f"{sgd_corrected_results_df[metric].std(ddof=1):.6f}"
    )


LEAKAGE-FREE SGD LOGISTIC - SEED 17
Iterations   : 100
Val AUC      : 0.683995
Test AUC     : 0.651203
Val AP       : 0.254708
Test AP      : 0.233804
Val Brier    : 0.124850
Test Brier   : 0.173074
Val LogLoss  : 0.406876
Test LogLoss : 0.518711

LEAKAGE-FREE SGD LOGISTIC - SEED 42
Iterations   : 90
Val AUC      : 0.679942
Test AUC     : 0.672703
Val AP       : 0.246350
Test AP      : 0.247242
Val Brier    : 0.123016
Test Brier   : 0.132301
Val LogLoss  : 0.401552
Test LogLoss : 0.425444

LEAKAGE-FREE SGD LOGISTIC - SEED 2026
Iterations   : 83
Val AUC      : 0.668687
Test AUC     : 0.664540
Val AP       : 0.234523
Test AP      : 0.239138
Val Brier    : 0.119579
Test Brier   : 0.122419
Val LogLoss  : 0.394700
Test LogLoss : 0.402627

LEAKAGE-FREE SGD LOGISTIC - THREE-SEED SUMMARY
 seed  val_auc  test_auc   val_ap  test_ap  val_brier  test_brier  val_logloss  test_logloss  n_iter
   17 0.683995  0.651203 0.254708 0.233804   0.124850    0.173074     0.406876      0.518711     100
   42 

In [47]:
# ============================================================
# LEAKAGE-FREE RERUN - Step 42
# Corrected HistGradientBoosting across 3 seeds
#
# These results supersede the provisional HistGB results.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)


histgb_corrected_results = []
histgb_corrected_predictions = {}


for seed in SEEDS:

    print("\n" + "=" * 75)
    print(
        f"LEAKAGE-FREE HISTGRADIENTBOOSTING - SEED {seed}"
    )
    print("=" * 75)

    model = HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        max_iter=300,
        max_depth=6,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=20,
        random_state=seed
    )

    model.fit(
        X_train_proc_corrected,
        y_train
    )


    p_val = model.predict_proba(
        X_val_proc_corrected
    )[:, 1]

    p_test = model.predict_proba(
        X_test_proc_corrected
    )[:, 1]


    p_val = np.clip(
        p_val,
        1e-7,
        1 - 1e-7
    )

    p_test = np.clip(
        p_test,
        1e-7,
        1 - 1e-7
    )


    histgb_corrected_predictions[
        seed
    ] = {
        "val": p_val.copy(),
        "test": p_test.copy()
    }


    row = {

        "seed":
            seed,

        "val_auc":
            roc_auc_score(
                y_val,
                p_val
            ),

        "test_auc":
            roc_auc_score(
                y_test,
                p_test
            ),

        "val_ap":
            average_precision_score(
                y_val,
                p_val
            ),

        "test_ap":
            average_precision_score(
                y_test,
                p_test
            ),

        "val_brier":
            brier_score_loss(
                y_val,
                p_val
            ),

        "test_brier":
            brier_score_loss(
                y_test,
                p_test
            ),

        "val_logloss":
            log_loss(
                y_val,
                p_val
            ),

        "test_logloss":
            log_loss(
                y_test,
                p_test
            ),

        "n_iter":
            int(
                model.n_iter_
            )
    }


    histgb_corrected_results.append(
        row
    )


    print(
        "Iterations   :",
        model.n_iter_
    )

    print(
        f"Val AUC      : "
        f"{row['val_auc']:.6f}"
    )

    print(
        f"Test AUC     : "
        f"{row['test_auc']:.6f}"
    )

    print(
        f"Val AP       : "
        f"{row['val_ap']:.6f}"
    )

    print(
        f"Test AP      : "
        f"{row['test_ap']:.6f}"
    )

    print(
        f"Val Brier    : "
        f"{row['val_brier']:.6f}"
    )

    print(
        f"Test Brier   : "
        f"{row['test_brier']:.6f}"
    )

    print(
        f"Val LogLoss  : "
        f"{row['val_logloss']:.6f}"
    )

    print(
        f"Test LogLoss : "
        f"{row['test_logloss']:.6f}"
    )


histgb_corrected_results_df = pd.DataFrame(
    histgb_corrected_results
)


print("\n" + "=" * 75)
print(
    "LEAKAGE-FREE HISTGB - THREE-SEED SUMMARY"
)
print("=" * 75)

print(
    histgb_corrected_results_df.to_string(
        index=False
    )
)


print("\nMEAN ± SD")

for metric in [
    "val_auc",
    "test_auc",
    "val_ap",
    "test_ap",
    "val_brier",
    "test_brier",
    "val_logloss",
    "test_logloss"
]:

    print(
        f"{metric:15s}: "
        f"{histgb_corrected_results_df[metric].mean():.6f} "
        f"± "
        f"{histgb_corrected_results_df[metric].std(ddof=1):.6f}"
    )


LEAKAGE-FREE HISTGRADIENTBOOSTING - SEED 17
Iterations   : 147
Val AUC      : 0.697693
Test AUC     : 0.695866
Val AP       : 0.269794
Test AP      : 0.279105
Val Brier    : 0.115986
Test Brier   : 0.118382
Val LogLoss  : 0.382419
Test LogLoss : 0.388596

LEAKAGE-FREE HISTGRADIENTBOOSTING - SEED 42
Iterations   : 152
Val AUC      : 0.698278
Test AUC     : 0.696749
Val AP       : 0.270221
Test AP      : 0.277673
Val Brier    : 0.115959
Test Brier   : 0.118359
Val LogLoss  : 0.382252
Test LogLoss : 0.388278

LEAKAGE-FREE HISTGRADIENTBOOSTING - SEED 2026
Iterations   : 153
Val AUC      : 0.697231
Test AUC     : 0.698576
Val AP       : 0.268654
Test AP      : 0.280272
Val Brier    : 0.116036
Test Brier   : 0.118195
Val LogLoss  : 0.382635
Test LogLoss : 0.387715

LEAKAGE-FREE HISTGB - THREE-SEED SUMMARY
 seed  val_auc  test_auc   val_ap  test_ap  val_brier  test_brier  val_logloss  test_logloss  n_iter
   17 0.697693  0.695866 0.269794 0.279105   0.115986    0.118382     0.382419      0.3

In [48]:
# ============================================================
# LEAKAGE-FREE RERUN - Step 43
# Corrected MLP across 3 seeds
#
# These results supersede the provisional MLP results.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)


mlp_corrected_results = []
mlp_corrected_predictions = {}


for seed in SEEDS:

    print("\n" + "=" * 75)
    print(
        f"LEAKAGE-FREE MLP - SEED {seed}"
    )
    print("=" * 75)

    model = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=512,
        learning_rate_init=0.001,
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=10,
        random_state=seed
    )

    model.fit(
        X_train_proc_corrected,
        y_train
    )


    p_val = model.predict_proba(
        X_val_proc_corrected
    )[:, 1]

    p_test = model.predict_proba(
        X_test_proc_corrected
    )[:, 1]


    p_val = np.clip(
        p_val,
        1e-7,
        1 - 1e-7
    )

    p_test = np.clip(
        p_test,
        1e-7,
        1 - 1e-7
    )


    mlp_corrected_predictions[
        seed
    ] = {
        "val": p_val.copy(),
        "test": p_test.copy()
    }


    row = {

        "seed":
            seed,

        "val_auc":
            roc_auc_score(
                y_val,
                p_val
            ),

        "test_auc":
            roc_auc_score(
                y_test,
                p_test
            ),

        "val_ap":
            average_precision_score(
                y_val,
                p_val
            ),

        "test_ap":
            average_precision_score(
                y_test,
                p_test
            ),

        "val_brier":
            brier_score_loss(
                y_val,
                p_val
            ),

        "test_brier":
            brier_score_loss(
                y_test,
                p_test
            ),

        "val_logloss":
            log_loss(
                y_val,
                p_val
            ),

        "test_logloss":
            log_loss(
                y_test,
                p_test
            ),

        "n_iter":
            int(
                model.n_iter_
            )
    }


    mlp_corrected_results.append(
        row
    )


    print(
        "Iterations   :",
        model.n_iter_
    )

    print(
        f"Val AUC      : "
        f"{row['val_auc']:.6f}"
    )

    print(
        f"Test AUC     : "
        f"{row['test_auc']:.6f}"
    )

    print(
        f"Val AP       : "
        f"{row['val_ap']:.6f}"
    )

    print(
        f"Test AP      : "
        f"{row['test_ap']:.6f}"
    )

    print(
        f"Val Brier    : "
        f"{row['val_brier']:.6f}"
    )

    print(
        f"Test Brier   : "
        f"{row['test_brier']:.6f}"
    )

    print(
        f"Val LogLoss  : "
        f"{row['val_logloss']:.6f}"
    )

    print(
        f"Test LogLoss : "
        f"{row['test_logloss']:.6f}"
    )


mlp_corrected_results_df = pd.DataFrame(
    mlp_corrected_results
)


print("\n" + "=" * 75)
print(
    "LEAKAGE-FREE MLP - THREE-SEED SUMMARY"
)
print("=" * 75)

print(
    mlp_corrected_results_df.to_string(
        index=False
    )
)


print("\nMEAN ± SD")

for metric in [
    "val_auc",
    "test_auc",
    "val_ap",
    "test_ap",
    "val_brier",
    "test_brier",
    "val_logloss",
    "test_logloss"
]:

    print(
        f"{metric:15s}: "
        f"{mlp_corrected_results_df[metric].mean():.6f} "
        f"± "
        f"{mlp_corrected_results_df[metric].std(ddof=1):.6f}"
    )


LEAKAGE-FREE MLP - SEED 17
Iterations   : 18
Val AUC      : 0.691163
Test AUC     : 0.657434
Val AP       : 0.260359
Test AP      : 0.246251
Val Brier    : 0.118550
Test Brier   : 0.134273
Val LogLoss  : 0.395653
Test LogLoss : 0.511628

LEAKAGE-FREE MLP - SEED 42
Iterations   : 13
Val AUC      : 0.693676
Test AUC     : 0.622032
Val AP       : 0.265649
Test AP      : 0.229636
Val Brier    : 0.117302
Test Brier   : 0.131930
Val LogLoss  : 0.387441
Test LogLoss : 0.488427

LEAKAGE-FREE MLP - SEED 2026
Iterations   : 15
Val AUC      : 0.691061
Test AUC     : 0.662319
Val AP       : 0.262254
Test AP      : 0.248491
Val Brier    : 0.116617
Test Brier   : 0.126867
Val LogLoss  : 0.384637
Test LogLoss : 0.431290

LEAKAGE-FREE MLP - THREE-SEED SUMMARY
 seed  val_auc  test_auc   val_ap  test_ap  val_brier  test_brier  val_logloss  test_logloss  n_iter
   17 0.691163  0.657434 0.260359 0.246251   0.118550    0.134273     0.395653      0.511628      18
   42 0.693676  0.622032 0.265649 0.229636 

In [49]:
# ============================================================
# LEAKAGE-FREE RERUN - Step 44
# Corrected paired DeLong AUC comparisons
#
# Uses the SAME test loans and matched seeds.
# These results supersede the provisional DeLong tests.
# ============================================================

import pandas as pd


corrected_comparison_rows = []

corrected_comparisons = [
    (
        "HistGB",
        histgb_corrected_predictions,
        "SGD Logistic",
        sgd_corrected_predictions
    ),
    (
        "HistGB",
        histgb_corrected_predictions,
        "MLP",
        mlp_corrected_predictions
    ),
    (
        "SGD Logistic",
        sgd_corrected_predictions,
        "MLP",
        mlp_corrected_predictions
    )
]


for seed in SEEDS:

    print("\n" + "=" * 80)
    print(
        f"LEAKAGE-FREE PAIRED DELONG TESTS - SEED {seed}"
    )
    print("=" * 80)

    for (
        name_a,
        preds_a,
        name_b,
        preds_b
    ) in corrected_comparisons:

        result = paired_delong(
            y_test,
            preds_a[seed]["test"],
            preds_b[seed]["test"]
        )

        corrected_comparison_rows.append({
            "seed": seed,
            "model_a": name_a,
            "model_b": name_b,
            **result
        })

        print(
            f"\n{name_a} vs {name_b}"
        )

        print(
            f"AUCs       : "
            f"{result['auc_a']:.6f} vs "
            f"{result['auc_b']:.6f}"
        )

        print(
            f"Delta AUC  : "
            f"{result['delta_auc']:.6f}"
        )

        print(
            f"95% CI     : "
            f"[{result['ci_low']:.6f}, "
            f"{result['ci_high']:.6f}]"
        )

        print(
            f"p-value    : "
            f"{result['p_value']:.6g}"
        )


corrected_delong_df = pd.DataFrame(
    corrected_comparison_rows
)


print("\n" + "=" * 80)
print(
    "LEAKAGE-FREE PAIRED DELONG SUMMARY"
)
print("=" * 80)

print(
    corrected_delong_df[
        [
            "seed",
            "model_a",
            "model_b",
            "auc_a",
            "auc_b",
            "delta_auc",
            "ci_low",
            "ci_high",
            "p_value"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print(
    "\nMEAN DELTA AUC ACROSS MATCHED SEEDS"
)

print(
    corrected_delong_df
    .groupby(
        [
            "model_a",
            "model_b"
        ]
    )["delta_auc"]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max"
        ]
    )
)


LEAKAGE-FREE PAIRED DELONG TESTS - SEED 17

HistGB vs SGD Logistic
AUCs       : 0.695866 vs 0.651203
Delta AUC  : 0.044663
95% CI     : [0.040743, 0.048584]
p-value    : 1.88998e-110

HistGB vs MLP
AUCs       : 0.695866 vs 0.657434
Delta AUC  : 0.038432
95% CI     : [0.034802, 0.042063]
p-value    : 1.29159e-95

SGD Logistic vs MLP
AUCs       : 0.651203 vs 0.657434
Delta AUC  : -0.006231
95% CI     : [-0.012204, -0.000258]
p-value    : 0.0408753

LEAKAGE-FREE PAIRED DELONG TESTS - SEED 42

HistGB vs SGD Logistic
AUCs       : 0.696749 vs 0.672703
Delta AUC  : 0.024046
95% CI     : [0.021316, 0.026776]
p-value    : 8.54076e-67

HistGB vs MLP
AUCs       : 0.696749 vs 0.622032
Delta AUC  : 0.074717
95% CI     : [0.070017, 0.079417]
p-value    : 4.25203e-213

SGD Logistic vs MLP
AUCs       : 0.672703 vs 0.622032
Delta AUC  : 0.050671
95% CI     : [0.045983, 0.055359]
p-value    : 1.34155e-99

LEAKAGE-FREE PAIRED DELONG TESTS - SEED 2026

HistGB vs SGD Logistic
AUCs       : 0.698576 vs 0.66

In [50]:
# ============================================================
# FINAL REVIEWER ANALYSIS - W2
# Corrected proper-score skill + validation-fitted recalibration
#
# This ONE cell completes W2.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def logit_safe(p, eps=1e-6):
    p = np.clip(
        np.asarray(p, dtype=float),
        eps,
        1 - eps
    )
    return np.log(
        p / (1 - p)
    )


def calibration_diagnostics(y, p):
    """
    Evaluation-only calibration intercept and slope.
    Ideal: intercept=0, slope=1.
    """
    z = logit_safe(
        p
    ).reshape(-1, 1)

    model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=1000
    )

    model.fit(
        z,
        y
    )

    return (
        float(model.intercept_[0]),
        float(model.coef_[0, 0])
    )


def quantile_ece(y, p, n_bins=10):

    y = np.asarray(y)
    p = np.asarray(p)

    order = np.argsort(p)
    bins = np.array_split(
        order,
        n_bins
    )

    ece = 0.0

    for idx in bins:

        ece += (
            len(idx) / len(y)
        ) * abs(
            y[idx].mean()
            - p[idx].mean()
        )

    return float(ece)


# ------------------------------------------------------------
# Corrected prediction dictionaries
# ------------------------------------------------------------

corrected_models = {
    "SGD Logistic":
        sgd_corrected_predictions,

    "HistGB":
        histgb_corrected_predictions,

    "MLP":
        mlp_corrected_predictions
}


# ------------------------------------------------------------
# Constant base-rate references
# ------------------------------------------------------------

val_prev = y_val.mean()
test_prev = y_test.mean()

val_base_p = np.full(
    len(y_val),
    val_prev
)

test_base_p = np.full(
    len(y_test),
    test_prev
)

val_brier0 = brier_score_loss(
    y_val,
    val_base_p
)

test_brier0 = brier_score_loss(
    y_test,
    test_base_p
)

val_ll0 = log_loss(
    y_val,
    val_base_p
)

test_ll0 = log_loss(
    y_test,
    test_base_p
)


print("=" * 100)
print("BASE-RATE REFERENCES")
print("=" * 100)

print(
    f"Validation prevalence : "
    f"{val_prev:.6f}"
)

print(
    f"Validation Brier_0    : "
    f"{val_brier0:.6f}"
)

print(
    f"Validation LogLoss_0  : "
    f"{val_ll0:.6f}"
)

print()

print(
    f"Test prevalence       : "
    f"{test_prev:.6f}"
)

print(
    f"Test Brier_0          : "
    f"{test_brier0:.6f}"
)

print(
    f"Test LogLoss_0        : "
    f"{test_ll0:.6f}"
)


# ------------------------------------------------------------
# Fit recalibration ONLY on validation predictions
# ------------------------------------------------------------

corrected_calibrated_predictions = {}

rows = []


for model_name, pred_dict in corrected_models.items():

    corrected_calibrated_predictions[
        model_name
    ] = {}

    for seed in SEEDS:

        p_val_raw = pred_dict[
            seed
        ]["val"]

        p_test_raw = pred_dict[
            seed
        ]["test"]


        # -----------------------------------------------
        # Fit recalibration on validation only
        # -----------------------------------------------

        z_val = logit_safe(
            p_val_raw
        ).reshape(-1, 1)

        z_test = logit_safe(
            p_test_raw
        ).reshape(-1, 1)


        calibrator = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=1000
        )

        calibrator.fit(
            z_val,
            y_val
        )


        p_val_cal = (
            calibrator
            .predict_proba(
                z_val
            )[:, 1]
        )

        p_test_cal = (
            calibrator
            .predict_proba(
                z_test
            )[:, 1]
        )


        corrected_calibrated_predictions[
            model_name
        ][seed] = {
            "val": p_val_cal.copy(),
            "test": p_test_cal.copy()
        }


        # -----------------------------------------------
        # Test proper scores
        # -----------------------------------------------

        raw_brier = brier_score_loss(
            y_test,
            p_test_raw
        )

        cal_brier = brier_score_loss(
            y_test,
            p_test_cal
        )

        raw_ll = log_loss(
            y_test,
            p_test_raw
        )

        cal_ll = log_loss(
            y_test,
            p_test_cal
        )


        # -----------------------------------------------
        # Test calibration diagnostics
        # -----------------------------------------------

        raw_intercept, raw_slope = (
            calibration_diagnostics(
                y_test,
                p_test_raw
            )
        )

        cal_intercept, cal_slope = (
            calibration_diagnostics(
                y_test,
                p_test_cal
            )
        )


        rows.append({

            "model":
                model_name,

            "seed":
                seed,

            "test_auc":
                roc_auc_score(
                    y_test,
                    p_test_raw
                ),

            "raw_brier":
                raw_brier,

            "raw_brier_skill":
                1 - raw_brier
                / test_brier0,

            "cal_brier":
                cal_brier,

            "cal_brier_skill":
                1 - cal_brier
                / test_brier0,

            "raw_logloss":
                raw_ll,

            "raw_logloss_skill":
                1 - raw_ll
                / test_ll0,

            "cal_logloss":
                cal_ll,

            "cal_logloss_skill":
                1 - cal_ll
                / test_ll0,

            "raw_cal_intercept":
                raw_intercept,

            "raw_cal_slope":
                raw_slope,

            "raw_ece":
                quantile_ece(
                    y_test,
                    p_test_raw
                ),

            "cal_cal_intercept":
                cal_intercept,

            "cal_cal_slope":
                cal_slope,

            "cal_ece":
                quantile_ece(
                    y_test,
                    p_test_cal
                )
        })


w2_corrected_df = pd.DataFrame(
    rows
)


print("\n" + "=" * 100)
print("W2 CORRECTED RESULTS - PER SEED")
print("=" * 100)

print(
    w2_corrected_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n" + "=" * 100)
print("W2 CORRECTED THREE-SEED MEAN ± SD")
print("=" * 100)


metrics = [
    "raw_brier_skill",
    "cal_brier_skill",
    "raw_logloss_skill",
    "cal_logloss_skill",
    "raw_cal_intercept",
    "raw_cal_slope",
    "raw_ece",
    "cal_cal_intercept",
    "cal_cal_slope",
    "cal_ece"
]


for model_name in w2_corrected_df[
    "model"
].unique():

    x = w2_corrected_df[
        w2_corrected_df[
            "model"
        ] == model_name
    ]

    print(
        f"\n{model_name}"
    )

    for metric in metrics:

        print(
            f"{metric:22s}: "
            f"{x[metric].mean():.6f} "
            f"± "
            f"{x[metric].std(ddof=1):.6f}"
        )

BASE-RATE REFERENCES
Validation prevalence : 0.144560
Validation Brier_0    : 0.123662
Validation LogLoss_0  : 0.413155

Test prevalence       : 0.148360
Test Brier_0          : 0.126349
Test LogLoss_0        : 0.419854


C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:


W2 CORRECTED RESULTS - PER SEED
       model  seed  test_auc  raw_brier  raw_brier_skill  cal_brier  cal_brier_skill  raw_logloss  raw_logloss_skill  cal_logloss  cal_logloss_skill  raw_cal_intercept  raw_cal_slope  raw_ece  cal_cal_intercept  cal_cal_slope  cal_ece
SGD Logistic    17  0.651203   0.173074        -0.369803   0.133518        -0.056735     0.518711          -0.235457     0.427842          -0.019027          -1.400255       0.523560 0.183249          -0.957909       0.619440 0.075353
SGD Logistic    42  0.672703   0.132301        -0.047107   0.121057         0.041884     0.425444          -0.013315     0.397667           0.052845          -0.871191       0.759752 0.089853          -0.206549       0.966156 0.019796
SGD Logistic  2026  0.664540   0.122419         0.031107   0.121190         0.040833     0.402627           0.041030     0.398572           0.050689          -0.508310       0.675755 0.025295          -0.039696       0.961842 0.006957
      HistGB    17  0.69586

C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\sufi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [51]:
# ============================================================
# FINAL REVIEWER ANALYSIS - W4
#
# Part A:
#   Governance threshold results for ALL THREE models
#   at r = 5, 10, 20
#
# Part B:
#   Bootstrap decision-cost continuum for selected HistGB
#   including validation threshold-selection variability
#
# After this cell, W4 is CLOSED.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

COST_RATIOS = np.array(
    [5, 10, 20],
    dtype=float
)

R_CURVE = np.array(
    [2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 20],
    dtype=float
)


def evaluate_cost(y, p, threshold, r):

    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    pred = (
        p >= threshold
    ).astype(int)

    tp = np.sum(
        (y == 1) & (pred == 1)
    )

    fp = np.sum(
        (y == 0) & (pred == 1)
    )

    fn = np.sum(
        (y == 1) & (pred == 0)
    )

    tn = np.sum(
        (y == 0) & (pred == 0)
    )

    n = len(y)

    prevalence = y.mean()

    raw_cost = (
        r * fn + fp
    ) / n

    approve_all_cost = (
        r * prevalence
    )

    reject_all_cost = (
        1 - prevalence
    )

    baseline = min(
        approve_all_cost,
        reject_all_cost
    )

    normalized_cost = (
        raw_cost / baseline
    )

    fnr = (
        fn / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    return {
        "normalized_cost":
            normalized_cost,

        "fnr":
            fnr,

        "fpr":
            fpr,

        "reject_rate":
            pred.mean()
    }


def exact_optimal_threshold(
    y,
    p,
    r
):

    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    order = np.argsort(-p)

    ps = p[order]
    ys = y[order]

    total_pos = ys.sum()

    cum_tp = np.cumsum(
        ys
    )

    cum_fp = np.cumsum(
        1 - ys
    )

    distinct_end = np.r_[
        ps[:-1] != ps[1:],
        True
    ]

    idx = np.where(
        distinct_end
    )[0]

    tp = cum_tp[idx]
    fp = cum_fp[idx]

    fn = total_pos - tp

    costs = (
        r * fn + fp
    )

    best = np.argmin(
        costs
    )

    none_cost = (
        r * total_pos
    )

    if none_cost < costs[best]:
        return 1.0

    return float(
        ps[idx[best]]
    )


def optimal_thresholds_many_r(
    y,
    p,
    r_values
):

    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    order = np.argsort(-p)

    ps = p[order]
    ys = y[order]

    total_pos = ys.sum()

    cum_tp = np.cumsum(
        ys
    )

    cum_fp = np.cumsum(
        1 - ys
    )

    distinct_end = np.r_[
        ps[:-1] != ps[1:],
        True
    ]

    idx = np.where(
        distinct_end
    )[0]

    thresholds = ps[idx]

    tp = cum_tp[idx]
    fp = cum_fp[idx]
    fn = total_pos - tp

    costs = (
        r_values[:, None]
        * fn[None, :]
        + fp[None, :]
    )

    best_idx = np.argmin(
        costs,
        axis=1
    )

    best_thresholds = (
        thresholds[
            best_idx
        ].astype(float)
    )

    best_costs = costs[
        np.arange(
            len(r_values)
        ),
        best_idx
    ]

    none_costs = (
        r_values
        * total_pos
    )

    use_none = (
        none_costs
        < best_costs
    )

    best_thresholds[
        use_none
    ] = 1.0

    return best_thresholds


# ------------------------------------------------------------
# A. ALL THREE MODELS: r = 5, 10, 20
# ------------------------------------------------------------

w4_rows = []


for model_name in [
    "SGD Logistic",
    "HistGB",
    "MLP"
]:

    for seed in SEEDS:

        p_val = (
            corrected_calibrated_predictions[
                model_name
            ][seed]["val"]
        )

        p_test = (
            corrected_calibrated_predictions[
                model_name
            ][seed]["test"]
        )

        for r in COST_RATIOS:

            theoretical_t = (
                1 / (1 + r)
            )

            val_t = (
                exact_optimal_threshold(
                    y_val,
                    p_val,
                    r
                )
            )

            theory = evaluate_cost(
                y_test,
                p_test,
                theoretical_t,
                r
            )

            valopt = evaluate_cost(
                y_test,
                p_test,
                val_t,
                r
            )


            w4_rows.append({

                "model":
                    model_name,

                "seed":
                    seed,

                "r":
                    int(r),

                "theoretical_t":
                    theoretical_t,

                "val_opt_t":
                    val_t,

                "theory_norm_cost":
                    theory[
                        "normalized_cost"
                    ],

                "valopt_norm_cost":
                    valopt[
                        "normalized_cost"
                    ],

                "theory_fnr":
                    theory["fnr"],

                "valopt_fnr":
                    valopt["fnr"],

                "theory_reject":
                    theory[
                        "reject_rate"
                    ],

                "valopt_reject":
                    valopt[
                        "reject_rate"
                    ]
            })


w4_corrected_df = pd.DataFrame(
    w4_rows
)


print("=" * 120)
print(
    "W4A - ALL THREE MODELS, "
    "CORRECTED THRESHOLD GOVERNANCE"
)
print("=" * 120)


w4_model_summary = (
    w4_corrected_df
    .groupby(
        [
            "model",
            "r"
        ]
    )
    .agg(
        val_opt_t_mean=(
            "val_opt_t",
            "mean"
        ),

        val_opt_t_sd=(
            "val_opt_t",
            "std"
        ),

        theory_norm_cost=(
            "theory_norm_cost",
            "mean"
        ),

        valopt_norm_cost=(
            "valopt_norm_cost",
            "mean"
        ),

        theory_fnr=(
            "theory_fnr",
            "mean"
        ),

        valopt_fnr=(
            "valopt_fnr",
            "mean"
        ),

        theory_reject=(
            "theory_reject",
            "mean"
        ),

        valopt_reject=(
            "valopt_reject",
            "mean"
        )
    )
    .reset_index()
)


print(
    w4_model_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# B. SELECTED HISTGB:
# bootstrap continuum with threshold-selection variability
# ============================================================

B = 300

rng = np.random.default_rng(
    26044
)

curve_rows = []

n_val = len(y_val)
n_test = len(y_test)

theory_thresholds = (
    1 / (1 + R_CURVE)
)


for b in range(B):

    val_idx = rng.integers(
        0,
        n_val,
        size=n_val
    )

    test_idx = rng.integers(
        0,
        n_test,
        size=n_test
    )

    yv = y_val[
        val_idx
    ]

    yt = y_test[
        test_idx
    ]

    seed_theory_cost = []
    seed_valopt_cost = []
    seed_thresholds = []


    for seed in SEEDS:

        pv = (
            corrected_calibrated_predictions[
                "HistGB"
            ][seed]["val"][
                val_idx
            ]
        )

        pt = (
            corrected_calibrated_predictions[
                "HistGB"
            ][seed]["test"][
                test_idx
            ]
        )

        opt_t = (
            optimal_thresholds_many_r(
                yv,
                pv,
                R_CURVE
            )
        )


        theory_costs = []
        valopt_costs = []


        for j, r in enumerate(
            R_CURVE
        ):

            theory_result = (
                evaluate_cost(
                    yt,
                    pt,
                    theory_thresholds[j],
                    r
                )
            )

            valopt_result = (
                evaluate_cost(
                    yt,
                    pt,
                    opt_t[j],
                    r
                )
            )

            theory_costs.append(
                theory_result[
                    "normalized_cost"
                ]
            )

            valopt_costs.append(
                valopt_result[
                    "normalized_cost"
                ]
            )


        seed_theory_cost.append(
            theory_costs
        )

        seed_valopt_cost.append(
            valopt_costs
        )

        seed_thresholds.append(
            opt_t
        )


    theory_mean = np.mean(
        seed_theory_cost,
        axis=0
    )

    valopt_mean = np.mean(
        seed_valopt_cost,
        axis=0
    )

    threshold_mean = np.mean(
        seed_thresholds,
        axis=0
    )


    for j, r in enumerate(
        R_CURVE
    ):

        curve_rows.append({

            "bootstrap":
                b + 1,

            "r":
                r,

            "theoretical_t":
                theory_thresholds[j],

            "val_opt_t":
                threshold_mean[j],

            "theory_norm_cost":
                theory_mean[j],

            "valopt_norm_cost":
                valopt_mean[j],

            "delta_valopt_minus_theory":
                valopt_mean[j]
                - theory_mean[j]
        })


    if (b + 1) % 50 == 0:

        print(
            f"Completed bootstrap "
            f"{b + 1}/{B}"
        )


w4_histgb_boot_df = pd.DataFrame(
    curve_rows
)


summary_rows = []


for r in R_CURVE:

    x = w4_histgb_boot_df[
        w4_histgb_boot_df[
            "r"
        ] == r
    ]

    summary_rows.append({

        "r":
            r,

        "theoretical_t":
            1 / (1 + r),

        "val_opt_t_mean":
            x[
                "val_opt_t"
            ].mean(),

        "val_opt_t_ci_low":
            x[
                "val_opt_t"
            ].quantile(
                0.025
            ),

        "val_opt_t_ci_high":
            x[
                "val_opt_t"
            ].quantile(
                0.975
            ),

        "theory_norm_cost":
            x[
                "theory_norm_cost"
            ].mean(),

        "theory_ci_low":
            x[
                "theory_norm_cost"
            ].quantile(
                0.025
            ),

        "theory_ci_high":
            x[
                "theory_norm_cost"
            ].quantile(
                0.975
            ),

        "valopt_norm_cost":
            x[
                "valopt_norm_cost"
            ].mean(),

        "valopt_ci_low":
            x[
                "valopt_norm_cost"
            ].quantile(
                0.025
            ),

        "valopt_ci_high":
            x[
                "valopt_norm_cost"
            ].quantile(
                0.975
            ),

        "delta_mean":
            x[
                "delta_valopt_minus_theory"
            ].mean(),

        "delta_ci_low":
            x[
                "delta_valopt_minus_theory"
            ].quantile(
                0.025
            ),

        "delta_ci_high":
            x[
                "delta_valopt_minus_theory"
            ].quantile(
                0.975
            ),

        "p_theory_below_1":
            np.mean(
                x[
                    "theory_norm_cost"
                ] < 1
            ),

        "p_valopt_below_1":
            np.mean(
                x[
                    "valopt_norm_cost"
                ] < 1
            )
    })


w4_histgb_summary_df = pd.DataFrame(
    summary_rows
)


print(
    "\n" + "=" * 140
)

print(
    "W4B - SELECTED HISTGB "
    "BOOTSTRAP DECISION-COST CONTINUUM"
)

print(
    "=" * 140
)


print(
    w4_histgb_summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

W4A - ALL THREE MODELS, CORRECTED THRESHOLD GOVERNANCE
       model  r  val_opt_t_mean  val_opt_t_sd  theory_norm_cost  valopt_norm_cost  theory_fnr  valopt_fnr  theory_reject  valopt_reject
      HistGB  5        0.167222      0.001442          0.772963          0.773408    0.470012    0.472644       0.303358       0.301346
      HistGB 10        0.096981      0.001196          0.860564          0.866485    0.141133    0.164389       0.650928       0.618017
      HistGB 20        0.052433      0.001169          0.951516          0.955095    0.028672    0.036036       0.869380       0.849485
         MLP  5        0.165849      0.004129          0.917978          0.916262    0.858589    0.855451       0.065034       0.066555
         MLP 10        0.093026      0.006400          1.206206          1.218859    0.544539    0.555446       0.286947       0.279923
         MLP 20        0.050470      0.001118          1.432154          1.481339    0.249190    0.271029       0.591672       0.

In [52]:
# ============================================================
# FINAL REVIEWER ANALYSIS - W5
#
# Selected model: corrected validation-recalibrated HistGB
# Prespecified slices: ALL 7 LendingClub grades A-G
# Operating point: r = 10, t* = 1/(1+r)
#
# Reports:
#   n
#   defaults
#   observed prevalence + bootstrap CI
#   mean predicted risk + bootstrap CI
#   calibration gap + bootstrap CI
#   normalized cost + bootstrap CI
#   FNR + bootstrap CI
#   rejection rate
#
# After this cell: STOP ALL ADDITIONAL ANALYSIS.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------

R_W5 = 10

T_W5 = 1 / (
    1 + R_W5
)

B_W5 = 500

rng_w5 = np.random.default_rng(
    26045
)


test_grade = (
    df_corrected.loc[
        df_corrected["split"] == "Test",
        "grade"
    ]
    .astype(str)
    .reset_index(drop=True)
    .to_numpy()
)


grade_levels = sorted(
    np.unique(test_grade)
)


print(
    "Prespecified grade slices:",
    grade_levels
)

print(
    "Number of slices examined:",
    len(grade_levels)
)

print(
    f"r={R_W5}, theoretical threshold="
    f"{T_W5:.6f}"
)


# ------------------------------------------------------------
# Point estimates + bootstrap
# ------------------------------------------------------------

point_rows = []
boot_rows = []


for grade in grade_levels:

    idx = np.where(
        test_grade == grade
    )[0]

    y_g = y_test[
        idx
    ]

    n_g = len(
        idx
    )

    defaults_g = int(
        y_g.sum()
    )

    # Mean across three matched HistGB seeds
    p_seed = [
        corrected_calibrated_predictions[
            "HistGB"
        ][seed]["test"][idx]
        for seed in SEEDS
    ]

    p_mean = np.mean(
        p_seed,
        axis=0
    )

    # Point estimate using seed-averaged probabilities
    cost_point = evaluate_cost(
        y_g,
        p_mean,
        T_W5,
        R_W5
    )

    observed_rate = (
        y_g.mean()
    )

    mean_pred = (
        p_mean.mean()
    )

    point_rows.append({

        "grade":
            grade,

        "n":
            n_g,

        "defaults":
            defaults_g,

        "observed_rate":
            observed_rate,

        "mean_pred":
            mean_pred,

        "calibration_gap":
            mean_pred
            - observed_rate,

        "normalized_cost":
            cost_point[
                "normalized_cost"
            ],

        "fnr":
            cost_point["fnr"],

        "rejection_rate":
            cost_point[
                "reject_rate"
            ]
    })


    # --------------------------------------------------------
    # Bootstrap WITHIN grade
    # Same resampled loans across all seeds
    # --------------------------------------------------------

    for b in range(B_W5):

        local_idx = rng_w5.integers(
            0,
            n_g,
            size=n_g
        )

        yb = y_g[
            local_idx
        ]

        p_seed_b = [
            p[
                local_idx
            ]
            for p in p_seed
        ]

        pb = np.mean(
            p_seed_b,
            axis=0
        )

        res = evaluate_cost(
            yb,
            pb,
            T_W5,
            R_W5
        )

        obs = (
            yb.mean()
        )

        pred = (
            pb.mean()
        )

        boot_rows.append({

            "grade":
                grade,

            "bootstrap":
                b + 1,

            "observed_rate":
                obs,

            "mean_pred":
                pred,

            "calibration_gap":
                pred - obs,

            "normalized_cost":
                res[
                    "normalized_cost"
                ],

            "fnr":
                res["fnr"],

            "rejection_rate":
                res[
                    "reject_rate"
                ]
        })


point_df = pd.DataFrame(
    point_rows
)

boot_df = pd.DataFrame(
    boot_rows
)


# ------------------------------------------------------------
# Build final manuscript-ready table
# ------------------------------------------------------------

final_rows = []


for grade in grade_levels:

    p = point_df[
        point_df["grade"]
        == grade
    ].iloc[0]

    b = boot_df[
        boot_df["grade"]
        == grade
    ]

    def q95(col):

        return (
            b[col].quantile(0.025),
            b[col].quantile(0.975)
        )


    obs_lo, obs_hi = q95(
        "observed_rate"
    )

    pred_lo, pred_hi = q95(
        "mean_pred"
    )

    gap_lo, gap_hi = q95(
        "calibration_gap"
    )

    cost_lo, cost_hi = q95(
        "normalized_cost"
    )

    fnr_lo, fnr_hi = q95(
        "fnr"
    )

    reject_lo, reject_hi = q95(
        "rejection_rate"
    )


    final_rows.append({

        "grade":
            grade,

        "n":
            int(p["n"]),

        "defaults":
            int(p["defaults"]),

        "observed_rate":
            p[
                "observed_rate"
            ],

        "obs_ci_low":
            obs_lo,

        "obs_ci_high":
            obs_hi,

        "mean_pred":
            p[
                "mean_pred"
            ],

        "pred_ci_low":
            pred_lo,

        "pred_ci_high":
            pred_hi,

        "calibration_gap":
            p[
                "calibration_gap"
            ],

        "gap_ci_low":
            gap_lo,

        "gap_ci_high":
            gap_hi,

        "normalized_cost":
            p[
                "normalized_cost"
            ],

        "cost_ci_low":
            cost_lo,

        "cost_ci_high":
            cost_hi,

        "fnr":
            p[
                "fnr"
            ],

        "fnr_ci_low":
            fnr_lo,

        "fnr_ci_high":
            fnr_hi,

        "rejection_rate":
            p[
                "rejection_rate"
            ],

        "reject_ci_low":
            reject_lo,

        "reject_ci_high":
            reject_hi,

        # Descriptive bootstrap probability
        # that the model policy beats
        # the best trivial within-grade policy
        "p_cost_below_1":
            np.mean(
                b[
                    "normalized_cost"
                ] < 1
            )
    })


w5_final_df = pd.DataFrame(
    final_rows
)


print(
    "\n" + "=" * 165
)

print(
    "W5 FINAL PREDECLARED GRADE-SLICE AUDIT "
    "- CORRECTED HISTGB"
)

print(
    "=" * 165
)


print(
    w5_final_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Prespecified grade slices: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Number of slices examined: 7
r=10, theoretical threshold=0.090909

W5 FINAL PREDECLARED GRADE-SLICE AUDIT - CORRECTED HISTGB
grade     n  defaults  observed_rate  obs_ci_low  obs_ci_high  mean_pred  pred_ci_low  pred_ci_high  calibration_gap  gap_ci_low  gap_ci_high  normalized_cost  cost_ci_low  cost_ci_high      fnr  fnr_ci_low  fnr_ci_high  rejection_rate  reject_ci_low  reject_ci_high  p_cost_below_1
    A 21784      1153       0.052929    0.050129     0.055799   0.051211     0.050866      0.051581        -0.001718   -0.004608     0.000985         0.975282     0.951885      1.000824 0.846487    0.825118     0.870196        0.076295       0.072320        0.079348        0.970000
    B 32135      3899       0.121332    0.117485     0.124740   0.112550     0.112126      0.112889        -0.008782   -0.012187    -0.004963         0.945814     0.925148      0.966180 0.199538    0.186431     0.212043        0.686074       0.68